# A Multi-Model Evaluation of LLM-Derived Sentiment for Magnificent Seven Stock Prediction

MSc Data Science individual project (INM363), City St George's, University of London. Author: Sirapob Charoenchaipiyakul. Supervisor: Oleksandr Galkin.

**Question.** Does adding news sentiment from FinBERT (a finance-specific encoder) or Qwen2.5-7B-Instruct (a general-purpose LLM) to a price, fundamental and macroeconomic baseline improve out-of-sample prediction and portfolio performance for the seven largest US technology stocks, and are any return differences statistically significant?

**Design.** Three feature sets (baseline, +FinBERT, +Qwen) are each used by three models (Ridge logistic regression, XGBoost, LSTM). Data, features, validation, hyperparameters and portfolio rules are held fixed, so only the sentiment input changes.

1. Data: prices, macroeconomic series, SEC fundamentals and 55,776 scored news headlines, 2010 to 2025 (Sections 2 to 5)
2. Exploratory analysis before any model is fitted (Section 6)
3. Purged walk-forward validation (2016 to 2025) and a weekly long-only top-3 portfolio (Sections 7 to 11)
4. Results: classification metrics, rank IC, return and Sharpe ratio (Section 12)
5. Significance tests on return (Sections 9 and 13)

**To run.** Cell 0 installs the dependencies. Downloads are cached in `data_cache/`, so reruns are offline. Set `SEC_USER_AGENT` in Section 1 before the first run (SEC requires a contact string). `QUICK_VERIFY = True` runs a fast smoke test, so set it to `False` for the full results. The reported results were produced on Google Colab with a T4 GPU.

## 0. Install dependencies

In [ ]:
!pip install -q yfinance optuna xgboost statsmodels mplfinance requests

## 1. Setup and configuration

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import io, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

TICKERS = ['AAPL', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NVDA', 'TSLA']

# ---------------------------------------------------------------------------
# Sample period. 16 years of data: the first year is used for feature warm-up
# (252-day windows), the next 5 years are the initial training window, and
# 2016-2025 is the walk-forward out-of-sample period. META and TSLA enter the
# pooled panel once their own warm-up is complete.
# ---------------------------------------------------------------------------
DATA_START = '2010-01-01'
DATA_END = '2025-12-31'
MODEL_START = '2011-01-03'          # first date a model row may exist (after warm-up)
INITIAL_TRAIN_YEARS = 5             # first OOS date = MODEL_START + 5y  (~2016-01)
TRAIN_WINDOW_YEARS = None           # None = expanding window. A number gives a rolling window

# Sentiment dates are taken from the headline file (Section 2.2) unless set by hand.
#  * SENTIMENT_START: first month with headlines. Earlier rows have sentiment = 0.
#  * The sentiment look-backs are chosen on the first SENTIMENT_SELECTION_YEARS year(s).
#  * Baseline-vs-sentiment comparisons start at SENTIMENT_EVAL_START (the end of the
#    selection year, never before the first out-of-sample date).
SENTIMENT_START = 'auto'            # or e.g. '2015-01-01'
SENTIMENT_SELECTION_YEARS = 1
SENTIMENT_EVAL_START = 'auto'       # or e.g. '2016-01-04'

HORIZON = 5                          # forward-return horizon = rebalance interval (days), see Section 2.3
PURGE_DAYS = HORIZON                 # training rows whose label overlaps the test window are dropped

# Label: 'cross_sectional' = 1 if the stock's 5-day forward return beats the same-day
# median of the 7 stocks, else 0. This is balanced by construction and removes the
# common market move. 'absolute' uses a plain up/down label instead.
LABEL_MODE = 'cross_sectional'
TUNING_METRIC = 'rank_ic'            # 'rank_ic' or 'macro_f1'

# Hyperparameters are tuned once per model family on the baseline feature set and reused for
# +FinBERT and +Qwen, so the only difference between the three models is the sentiment inputs.
# Tuning each feature set separately gave different settings and the comparison then mostly
# reflected tuning noise. Set False to tune each feature set on its own.
SHARED_HYPERPARAMS = True

# How headlines are assigned to trading sessions.
#  'strict'    : a headline informs the first session strictly after its calendar date, for all years.
#                This is the only rule that is the same across the whole sample, because part of the
#                file has no publication times.
#  'timestamp' : use the file's effective_date column (for example published before 15:45 ET -> same day).
SENTIMENT_ALIGNMENT = 'strict'

QUICK_VERIFY = False   # True = fast smoke test with small settings, False = full run

if QUICK_VERIFY:
    RETRAIN_STEP_DAYS = 504
    N_TRIALS = 3
    LSTM_TUNING_EPOCHS = 2
    LSTM_EPOCHS = 3
else:
    RETRAIN_STEP_DAYS = 126          # retrain about every 6 months
    N_TRIALS = 20
    LSTM_TUNING_EPOCHS = 15
    LSTM_EPOCHS = 20

TRANSACTION_COST_BPS = 5

# ---------------------------------------------------------------------------
# Look-back selection, see Section 3
# ---------------------------------------------------------------------------
MAX_AR_LAG = 20          # largest AR order considered by BIC
MAX_LOOKBACK = 126       # hard cap on any data-driven window (~6 months)
IRF_TOL = 0.10           # a shock is "forgotten" once |impulse response| < 10% of its size
ARIMA_MAX_P, ARIMA_MAX_Q = 2, 2
VIX_REGIME_QUANTILE = 0.5   # high-VIX regime = above the training-window median VIX
LSTM_SEQ_BOUNDS = (5, 42)   # guardrails on the data-driven LSTM sequence length

# ---------------------------------------------------------------------------
# Portfolio construction (Section 7a)
# ---------------------------------------------------------------------------
# 'rank'      : each rebalance date, hold the K stocks with the highest model score.
#              The book is always invested, apart from VIX scaling and the drawdown breaker.
# 'threshold' : a stock is eligible only if its probability clears a rolling threshold.
# All reported results use 'rank'. The threshold code is still run for every model but is
# not used by build_position_weights in this mode, so it does not affect any result.
PORTFOLIO_MODE = 'rank'
PORTFOLIO_TOP_K = 3
PORTFOLIO_MAX_WEIGHT = 0.50      # per-name cap; weights are inverse-vol, gross <= 1
VIX_PCTILE_SOFT = 0.70           # full size while VIX is below its own expanding 70th pct.
VIX_PCTILE_HARD = 0.95           # exposure floors at VIX_MIN_SCALE at the 95th pct.
VIX_MIN_SCALE = 0.3
DRAWDOWN_LIMIT = 0.15
DRAWDOWN_RECOVERY = 0.07
DRAWDOWN_SCALE = 0.0
MIN_SIGNAL_PERCENTILE = 0.85
MIN_RELIABLE_TRADES = 5
LONG_ONLY_MODE = True
THRESHOLD_LOOKBACK_DAYS = 252    # thresholds are re-selected at each retrain on the prior year
THRESHOLD_MIN_DAYS = 120         # of out-of-sample predictions (avoids threshold leakage)

EARLY_STOP_VAL_FRAC = 0.15
EARLY_STOPPING_ROUNDS = 20
EARLY_STOP_PATIENCE = 5

# ---------------------------------------------------------------------------
# Data access
# ---------------------------------------------------------------------------
SEC_USER_AGENT = "Your Name your@email.com"   # SEC EDGAR requires a name and email here
USE_FUNDAMENTALS = True
CACHE_DIR = 'data_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"QUICK_VERIFY={QUICK_VERIFY} | data {DATA_START}..{DATA_END} | label={LABEL_MODE} | "
      f"retrain every {RETRAIN_STEP_DAYS}d | purge={PURGE_DAYS}d | N_TRIALS={N_TRIALS}")
import torch
print(f"GPU available: {torch.cuda.is_available()}")


## 1a. Metric definitions

**Cumulative return**: total compounded growth of the strategy's capital over the evaluation window, net of transaction costs.

**Annualised return, volatility, Sharpe ratio and maximum drawdown**: standard strategy metrics. Sharpe = mean period return / standard deviation of period return x sqrt(periods per year), with 252/5 five-day periods per year. `sharpe` uses raw returns (risk-free rate = 0). `sharpe_excess` subtracts the 3-month T-bill rate (FRED DTB3, lagged one business day), prorated to each 5-day period. Idle cash earns no interest, so it is slightly conservative for strategies that are not fully invested. Maximum drawdown is the worst peak-to-trough loss of the equity curve.

**Rank IC (information coefficient)**: on each rebalance date, the Spearman correlation between the model score and the realised 5-day forward return across the 7 stocks, averaged over dates. It measures whether the model ranks stocks correctly, independent of any sizing rule. The t-statistic uses Newey-West (HAC) standard errors.

**Active period, hit rate and mean return per active trade**: an active period is a rebalance date on which the portfolio held something. Hit rate is the share of active periods with a positive net return. A period in which the drawdown breaker forces the portfolio into cash looks the same as a period with no holdings.

**Evaluation windows.** Results are reported for `full_oos` (the 10-year walk-forward from `FIRST_OOS_DATE`). A second window, `sentiment_era`, is added only when sentiment starts after the first out-of-sample date. With the 2015 to 2025 headlines the two coincide, so only `full_oos` is used.

**Confidence interval and p-value.** A block-bootstrap interval shows the size of an effect, and a paired block-permutation test shows whether it is significant.

## 1b. Key design choices

| Choice | What the code does | Section |
|---|---|---|
| Label | 1 if the stock's 5-day forward return beats the same-day median of the 7 stocks, which removes the common market move | 1, 4 |
| Features | 56 baseline features: price, point-in-time fundamentals (SEC XBRL) and macro and VIX state variables | 4, 5 |
| Look-backs | Estimated on pre-2016 data with ADF/KPSS tests, AR(p) selected by BIC and impulse-response decay, separately in low-VIX and high-VIX regimes | 3 |
| Validation | Expanding-window walk-forward with a 5-day purge between training and test, refit every 126 days (20 refits) | 7b |
| Hyperparameters | Tuned once per model family on the baseline features (Optuna, 20 trials, data before 2016) and reused for +FinBERT and +Qwen | 7b, 10 |
| Portfolio | Weekly long-only top 3 by predicted probability, inverse-volatility weights (50% cap), VIX scaling, 15% drawdown breaker, 5 bp costs | 7a |
| Significance | Paired block-permutation test on cumulative return (10-period blocks, 2,000 permutations) | 9, 13 |

## 1c. Where each method comes from

**Literature** = taken from the cited source. **Adapted** = the idea is from the literature and the specific implementation is mine. **Own design** = not from a specific source, and the reasons are given in the linked section.

| Component | Status | Main references | Details |
|---|---|---|---|
| Purged walk-forward and purged cross-validation | Literature | Jansen (2020), `MultipleTimeSeriesCV` with a `lookahead` purge | 7b |
| Cross-sectional label, rank IC with HAC t-statistics | Literature | Jansen (2020) | 1a, 12 |
| Momentum, trend, risk and liquidity features | Literature | Jansen (2020), Jegadeesh & Titman (1993), Amihud (2002) | 4 |
| Point-in-time fundamentals, macro publication lag | Literature | Jansen (2020) | 2.1 |
| Fundamentals ranked across stocks | Literature | Gu, Kelly & Xiu (2020) | 4 |
| SUE earnings surprise | Literature | Foster, Olsen & Shevlin (1984) | 2.1 |
| ADF/KPSS stationarity, AR order by BIC, ARIMA forecast as a feature | Literature | Box & Jenkins (1970), Jansen (2020) | 3 |
| Look-back = max(p, impulse-response decay), 10% tolerance, cap | Own design (on standard tools) | Hamilton (1994), Chan (2013) | 3.0 |
| VIX-regime-specific look-backs | Adapted | LeBaron (1992), Garcia (2013), Hamilton (1989) | 3.0 |
| Sentiment lagged one day, weekend headlines rolled forward | Literature | Jansen (2020) | 2.2 |
| All features given to the models, selection by regularisation | Literature | Gu, Kelly & Xiu (2020) | 5.1 |
| Top-K ranking portfolio, inverse-volatility weights | Literature | Jansen (2020), Maillard et al. (2010) | 7a |
| VIX-percentile exposure scaling (70th to 95th percentile, 30% floor) | Adapted | Moreira & Muir (2017), Barroso & Santa-Clara (2015), Daniel & Moskowitz (2016) | 3.0, 7a |
| Drawdown circuit breaker | Practitioner rule | None | 7a |
| 10-year out-of-sample period, 5-year initial training window | Adapted | Jansen (2020) case studies | 1 |

## 2. Data acquisition (16 years, cached)

Everything is downloaded once and cached in `data_cache/`. Sources:

* **Prices** (7 stocks, S&P 500 `^GSPC`, VIX `^VIX`, 3-month VIX `^VIX3M`): Yahoo Finance via `yfinance`, split- and dividend-adjusted, plus the split history (needed to put EPS on the same share basis as adjusted prices).
* **Macro**: FRED CSV endpoint (no API key).
* **Fundamentals**: SEC EDGAR XBRL `companyfacts` API. Each quarterly value is taken **as first reported** (earliest filing) and stamped with its filing date. Fiscal Q4 is derived as annual minus Q1 to Q3 and stamped with the 10-K filing date. Alphabet uses both its own CIK and Google Inc.'s pre-2015 CIK.
* **Sentiment**: `mag7_combined_dataset.csv` (2015 to 2025), the news headlines with their FinBERT and Qwen scores. All technical features are computed from the 16-year price history.

In [ ]:
import requests

def _find_file(filename):
    candidates = [filename, os.path.join('/content', filename),
                  os.path.join('/content/drive/MyDrive', filename),
                  os.path.join('/mnt/user-data/uploads', filename)]
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f"Could not find '{filename}' in any of: {candidates}.")

def _cache(name):
    return os.path.join(CACHE_DIR, name)

SENTIMENT_PATH = _find_file('mag7_combined_dataset.csv')

# ---------------------------------------------------------------- Yahoo Finance
def download_yf_history(symbols, start, end):
    import yfinance as yf
    frames, splits = [], []
    end_excl = (pd.Timestamp(end) + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
    for s in symbols:
        h = yf.Ticker(s).history(start=start, end=end_excl, auto_adjust=True, actions=True)
        if h is None or h.empty:
            raise RuntimeError(f"yfinance returned no data for {s}")
        h.index = pd.to_datetime(h.index).tz_localize(None).normalize()
        h = h.rename_axis('date').reset_index()
        h['symbol'] = s
        frames.append(h[['date', 'symbol', 'Open', 'High', 'Low', 'Close', 'Volume']])
        if 'Stock Splits' in h.columns:
            sp = h.loc[h['Stock Splits'] > 0, ['date', 'Stock Splits']].rename(columns={'Stock Splits': 'ratio'})
            sp['symbol'] = s
            splits.append(sp)
        time.sleep(0.3)
    sp_all = pd.concat(splits, ignore_index=True) if splits else pd.DataFrame(columns=['date', 'ratio', 'symbol'])
    return pd.concat(frames, ignore_index=True), sp_all

def load_prices():
    f_eq, f_sp, f_mk = _cache('equity_ohlcv.csv'), _cache('splits.csv'), _cache('market_ohlcv.csv')
    if not (os.path.exists(f_eq) and os.path.exists(f_sp)):
        eq, sp = download_yf_history(TICKERS, DATA_START, DATA_END)
        eq.to_csv(f_eq, index=False); sp.to_csv(f_sp, index=False)
    if not os.path.exists(f_mk):
        mk, _ = download_yf_history(['^GSPC', '^VIX', '^VIX3M'], DATA_START, DATA_END)
        mk.to_csv(f_mk, index=False)
    eq = pd.read_csv(f_eq, parse_dates=['date'])
    sp = pd.read_csv(f_sp, parse_dates=['date'])
    mk = pd.read_csv(f_mk, parse_dates=['date'])
    return eq, sp, mk

# ---------------------------------------------------------------- FRED
FRED_SERIES = {
    'DGS10': 'ust10y',            # 10Y Treasury yield
    'DTB3': 'tbill3m',            # 3M T-bill
    'BAA10Y': 'credit_spread',    # Moody's Baa - 10Y Treasury
    'T10YIE': 'breakeven10y',     # 10Y breakeven inflation (inflation expectations)
    'DTWEXBGS': 'usd_broad',      # broad trade-weighted dollar
    'DCOILBRENTEU': 'brent',      # Brent crude (never negative, unlike WTI in Apr-2020)
}

def download_fred(series_ids, start, end):
    out = []
    for sid in series_ids:
        url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}&cosd={start}&coed={end}'
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        s = pd.read_csv(io.StringIO(r.text))
        s.columns = ['date', sid]
        s['date'] = pd.to_datetime(s['date'])
        s[sid] = pd.to_numeric(s[sid], errors='coerce')     # FRED uses '.' for missing
        out.append(s.set_index('date'))
        time.sleep(0.3)
    return pd.concat(out, axis=1).sort_index()

def load_fred():
    f = _cache('fred.csv')
    if not os.path.exists(f):
        download_fred(list(FRED_SERIES), DATA_START, DATA_END).reset_index().to_csv(f, index=False)
    return pd.read_csv(f, parse_dates=['date']).set_index('date').rename(columns=FRED_SERIES)

# ---------------------------------------------------------------- SEC XBRL
SEC_CIKS = {'AAPL': [320193], 'AMZN': [1018724], 'GOOGL': [1652044, 1288776],
            'META': [1326801], 'MSFT': [789019], 'NVDA': [1045810], 'TSLA': [1318605]}
SEC_CONCEPTS = {   # field: (concepts in priority order, unit, period type)
    'eps': (['EarningsPerShareDiluted', 'EarningsPerShareBasic'], 'USD/shares', 'duration'),
    'revenue': (['Revenues', 'RevenueFromContractWithCustomerExcludingAssessedTax',
                 'SalesRevenueNet', 'RevenueFromContractWithCustomerIncludingAssessedTax'], 'USD', 'duration'),
    'net_income': (['NetIncomeLoss'], 'USD', 'duration'),
    'equity': (['StockholdersEquity',
                'StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest'], 'USD', 'instant'),
}

def fetch_sec_facts_raw():
    if 'example.com' in SEC_USER_AGENT:
        raise ValueError("Set SEC_USER_AGENT in Section 1 to 'Your Name your@email' -- SEC rejects anonymous requests.")
    rows = []
    for tk, ciks in SEC_CIKS.items():
        for cik in ciks:
            url = f'https://data.sec.gov/api/xbrl/companyfacts/CIK{cik:010d}.json'
            r = requests.get(url, headers={'User-Agent': SEC_USER_AGENT}, timeout=60)
            r.raise_for_status()
            facts = r.json().get('facts', {}).get('us-gaap', {})
            for field, (concepts, unit, _) in SEC_CONCEPTS.items():
                for pri, c in enumerate(concepts):
                    for f in facts.get(c, {}).get('units', {}).get(unit, []):
                        if f.get('form') not in ('10-Q', '10-K'):
                            continue
                        rows.append(dict(ticker=tk, field=field, concept=c, priority=pri,
                                         start=f.get('start'), end=f['end'], val=f['val'],
                                         filed=f['filed'], form=f['form']))
            time.sleep(0.25)   # SEC fair-access limit is 10 req/s
    return pd.DataFrame(rows)

def load_sec_facts_raw():
    f = _cache('sec_facts_raw.csv')
    if not os.path.exists(f):
        fetch_sec_facts_raw().to_csv(f, index=False)
    return pd.read_csv(f, parse_dates=['start', 'end', 'filed'])

eq_raw, splits_raw, mkt_raw = load_prices()
fred_raw = load_fred()
sec_raw = load_sec_facts_raw() if USE_FUNDAMENTALS else None
print(f"Equities: {eq_raw['date'].min().date()} .. {eq_raw['date'].max().date()} "
      f"({eq_raw['symbol'].nunique()} tickers, {len(eq_raw):,} rows)")
print(eq_raw.groupby('symbol')['date'].agg(['min', 'max', 'count']))
print(f"Market series: {sorted(mkt_raw['symbol'].unique())} | FRED: {list(fred_raw.columns)}")
if sec_raw is not None:
    print(f"SEC XBRL facts: {len(sec_raw):,} rows")

### 2.1 Point-in-time quarterly fundamentals

The rules that keep fundamentals free of look-ahead (Jansen, 2020):

1. **As first reported.** Each quarter's value is the one in the *earliest* filing that reports it. Later comparatives and restatements are ignored.
2. **Available on filing date + 1 business day**, not on the fiscal period end (filings often land after the close).
3. **Split-consistent EPS.** Yahoo prices are adjusted for every later split, so EPS reported before a split is divided by the product of split ratios that occur after its filing date. This is a unit conversion (both sides of earnings yield use today's share basis) and not information about the future.
4. Fiscal Q4 = annual - (Q1 + Q2 + Q3), because 10-Ks report only the annual figure.

In [ ]:
def build_quarterly_fundamentals(raw, splits):
    raw = raw.copy()
    long_rows = []
    for (tk, field), g in raw.groupby(['ticker', 'field']):
        kind = SEC_CONCEPTS[field][2]
        if kind == 'instant':
            g = g.sort_values(['end', 'filed', 'priority']).groupby('end', as_index=False).first()
            long_rows.append(g.assign(ticker=tk, field=field)[['ticker', 'field', 'end', 'filed', 'val']])
            continue
        g = g.dropna(subset=['start']).copy()
        g['days'] = (g['end'] - g['start']).dt.days
        # first-reported value per (start, end); concept priority breaks ties within a filing
        g = g.sort_values(['start', 'end', 'filed', 'priority']).groupby(['start', 'end'], as_index=False).first()
        q = g[(g['days'] >= 80) & (g['days'] <= 100)]
        a = g[(g['days'] >= 350) & (g['days'] <= 380)]
        derived = []
        for _, ar in a.iterrows():
            tol = pd.Timedelta(days=7)
            inside = q[(q['start'] >= ar['start'] - tol) & (q['end'] <= ar['end'] + tol)]
            if (inside['end'] >= ar['end'] - tol).any():     # Q4 already reported as a quarter
                continue
            if len(inside) == 3:
                derived.append(dict(start=inside['end'].max() + pd.Timedelta(days=1), end=ar['end'],
                                    filed=ar['filed'], val=ar['val'] - inside['val'].sum()))
        qq = pd.concat([q[['start', 'end', 'filed', 'val']], pd.DataFrame(derived)], ignore_index=True)
        qq = qq.sort_values(['end', 'filed']).drop_duplicates('end', keep='first')
        long_rows.append(qq.assign(ticker=tk, field=field)[['ticker', 'field', 'end', 'filed', 'val']])
    long = pd.concat(long_rows, ignore_index=True)

    # one row per (ticker, fiscal quarter end); availability = LAST of the fields' filing dates
    wide = long.pivot_table(index=['ticker', 'end'], columns='field', values='val', aggfunc='first')
    filed = long[long['field'] != 'equity'].groupby(['ticker', 'end'])['filed'].max()
    wide = wide.join(filed).reset_index().sort_values(['ticker', 'end'])
    wide = wide.dropna(subset=['filed', 'eps'])

    # split adjustment of EPS to today's share basis
    def later_split_factor(tk, filed_date):
        s = splits[(splits['symbol'] == tk) & (splits['date'] > filed_date)]
        return float(np.prod(s['ratio'])) if len(s) else 1.0
    wide['split_factor'] = [later_split_factor(t, f) for t, f in zip(wide['ticker'], wide['filed'])]
    wide['eps'] = wide['eps'] / wide['split_factor']

    def sym_growth(a, b):
        den = (a.abs() + b.abs()) / 2
        return ((a - b) / den.where(den > 0)).clip(-2, 2)

    out = []
    for tk, g in wide.groupby('ticker'):
        g = g.sort_values('end').copy()
        for c in ['eps', 'revenue', 'net_income']:
            if c not in g:
                g[c] = np.nan
            g[f'{c}_ttm'] = g[c].rolling(4, min_periods=4).sum()
        g['eps_growth_yoy'] = sym_growth(g['eps_ttm'], g['eps_ttm'].shift(4))
        g['rev_growth_yoy'] = g['revenue_ttm'] / g['revenue_ttm'].shift(4) - 1
        g['roe'] = g['net_income_ttm'] / g['equity'].where(g['equity'] > 0) if 'equity' in g else np.nan
        g['net_margin'] = g['net_income_ttm'] / g['revenue_ttm'].where(g['revenue_ttm'] > 0)
        # SUE (Foster, Olsen & Shevlin 1984): seasonal random-walk surprise / std of prior surprises
        surprise = g['eps'] - g['eps'].shift(4)
        g['sue'] = (surprise / surprise.shift(1).rolling(8, min_periods=4).std()).clip(-10, 10)
        out.append(g)
    fq = pd.concat(out, ignore_index=True)
    fq['available_date'] = fq['filed'] + pd.offsets.BDay(1)
    return fq

if USE_FUNDAMENTALS:
    fund_q = build_quarterly_fundamentals(sec_raw, splits_raw)
    print(f"{len(fund_q):,} ticker-quarters | first available per ticker:")
    print(fund_q.groupby('ticker')['available_date'].min().dt.date)
    print(fund_q[['ticker', 'end', 'filed', 'eps', 'eps_ttm', 'eps_growth_yoy', 'rev_growth_yoy',
                  'roe', 'net_margin', 'sue']].tail(8).round(3))
else:
    fund_q = None

### 2.2 Sentiment: trading-date alignment

A headline is only usable once it is known, so each headline is assigned to the first trading session strictly after its calendar date (`SENTIMENT_ALIGNMENT = 'strict'`, the default). This lags every headline by one day in every year, because part of the file has date-only timestamps and one rule has to apply throughout (Jansen, 2020).

The alternative, `'timestamp'`, uses the file's `effective_date` (published before 15:45 New York time on a trading day = that day, otherwise the next session). It is only suitable when every year was built with the same rule. The loader prints the same-day share per year and warns if it differs by more than 30 points between years.

Weekend and holiday headlines roll forward to the next session instead of being dropped, and exact duplicate (ticker, date, headline) rows are removed. The sentiment dates (`SENTIMENT_START`, selection year, `SENTIMENT_EVAL_START`) are then set from the data.

In [ ]:
def load_sentiment_daily(trading_dates):
    s = pd.read_csv(SENTIMENT_PATH, parse_dates=['date'])
    n0 = len(s)
    s = s.drop_duplicates(subset=['target_ticker', 'date', 'headline'])
    if SENTIMENT_START != 'auto':          # set by hand: use only headlines from that date on (a consistent-source sub-sample)
        s = s[s['date'] >= pd.Timestamp(SENTIMENT_START)]
    td = np.array(sorted(pd.to_datetime(trading_dates).unique()), dtype='datetime64[ns]')
    # fallback for date-only rows: first session strictly after the calendar date
    idx = np.searchsorted(td, s['date'].values.astype('datetime64[ns]'), side='right')
    fallback = pd.Series(pd.NaT, index=s.index, dtype='datetime64[ns]')
    fallback[idx < len(td)] = td[idx[idx < len(td)]]
    if SENTIMENT_ALIGNMENT == 'timestamp' and 'effective_date' in s.columns:
        eff = pd.to_datetime(s['effective_date'], errors='coerce')
        has_ts = eff.notna() & np.isin(eff.values.astype('datetime64[ns]'), td)
        # a file built with two different rules shows up as a jump in the same-day share between years
        _sd = (eff == s['date']).groupby(s['date'].dt.year).mean().mul(100).round(1)
        print("Share of headlines assigned to their own publication day, by year (%):", _sd.to_dict())
        if _sd.max() - _sd.min() > 30:
            print("WARNING: the same-day share differs by more than 30 points between years, so the file mixes alignment rules "
                  "(features would be up to a day fresher in some years). Use SENTIMENT_ALIGNMENT = 'strict'.")
    else:
        eff, has_ts = fallback, pd.Series(False, index=s.index)     # strict next-session rule for every headline
    s['eff_date'] = np.where(has_ts, eff, fallback)
    s['eff_date'] = pd.to_datetime(s['eff_date'])
    s = s[s['eff_date'].notna()].copy()
    has_ts = has_ts.loc[s.index]
    print(f"Session alignment: {has_ts.mean():.1%} of headlines from publication timestamps, "
          f"{(~has_ts).mean():.1%} by the date-only rule")
    on_trading_day = np.isin(s['date'].values.astype('datetime64[ns]'), td)
    daily = s.groupby(['target_ticker', 'eff_date']).agg(
        finbert_sum=('finbert_sentiment_score', 'sum'),
        qwen_sum=('qwen_sentiment_score', 'sum'),
        n_headlines=('headline', 'count')).reset_index().rename(
        columns={'target_ticker': 'ticker', 'eff_date': 'date'})
    print(f"Headlines: {n0:,} raw, {len(s):,} after de-duplication; "
          f"{(~on_trading_day).sum():,} dated on non-trading days (now rolled forward, "
          f"previously dropped by the date merge)")
    assert (s.loc[has_ts, 'eff_date'] >= s.loc[has_ts, 'date']).all(), "effective date before publication date"
    assert (s.loc[~has_ts, 'eff_date'] > s.loc[~has_ts, 'date']).all(), "date-only headlines must be lagged a day"
    return daily, s

trading_dates = np.sort(eq_raw['date'].unique())
sent_daily, sent_headlines = load_sentiment_daily(trading_dates)

# ---- sentiment dates from the data (unless set by hand in Section 1) ----
_first_oos_approx = pd.DatetimeIndex(trading_dates)[
    pd.DatetimeIndex(trading_dates) >= pd.Timestamp(MODEL_START) + pd.DateOffset(years=INITIAL_TRAIN_YEARS)][0]
if SENTIMENT_START == 'auto':
    SENTIMENT_START = sent_daily['date'].min().to_period('M').to_timestamp().strftime('%Y-%m-%d')
SENTIMENT_SELECTION_END = (pd.Timestamp(SENTIMENT_START) + pd.DateOffset(years=SENTIMENT_SELECTION_YEARS)).strftime('%Y-%m-%d')
if SENTIMENT_EVAL_START == 'auto':
    SENTIMENT_EVAL_START = max(pd.Timestamp(SENTIMENT_SELECTION_END), _first_oos_approx).strftime('%Y-%m-%d')
print(f"SENTIMENT_START={SENTIMENT_START} | look-backs selected on [{SENTIMENT_START}, {SENTIMENT_SELECTION_END}) "
      f"| sentiment comparisons from {SENTIMENT_EVAL_START}")
print(sent_daily.groupby(sent_daily['date'].dt.year)['n_headlines'].sum().rename('headlines per year').to_string())

# ---- coverage by year: a level shift here is a data-pipeline signal, not a market one ----
_yr = sent_daily['date'].dt.year
_cov = pd.DataFrame({'headlines': sent_daily.groupby(_yr)['n_headlines'].sum(), 'ticker_sessions': sent_daily.groupby(_yr).size()})
_tdays = pd.Series(pd.DatetimeIndex(trading_dates).year).value_counts().sort_index()
_cov['% of ticker-sessions with news'] = (_cov['ticker_sessions'] / (_tdays.reindex(_cov.index) * len(TICKERS)) * 100).round(1)
_cov['headlines per ticker-session with news'] = (_cov['headlines'] / _cov['ticker_sessions']).round(2)
print("\nSentiment coverage by year:")
print(_cov.drop(columns='ticker_sessions').to_string())
_ratio = (_cov['headlines'] / _cov['headlines'].shift(1)).dropna()
_breaks = _ratio[(_ratio > 1.6) | (_ratio < 1 / 1.6)]
if len(_breaks):
    print(f"WARNING: headline volume changes by more than 60% between consecutive years at {[int(y) for y in _breaks.index]} "
          f"(ratios {[round(float(r), 2) for r in _breaks.values]}). If this coincides with the boundary between two separately built "
          "headline files, the has-news flags and the averages are not comparable across it: rebuild both halves with the same "
          "pipeline, or set SENTIMENT_START to the start of the consistent part.")
_c = _cov['% of ticker-sessions with news']
if len(_c) >= 6 and abs(_c.iloc[-3:].mean() - _c.iloc[:3].mean()) > 15:
    print(f"WARNING: the share of ticker-sessions with news moves from {_c.iloc[:3].mean():.0f}% (first three years) to {_c.iloc[-3:].mean():.0f}% "
          f"(last three years), a level shift of more than 15 points even if no single year jumps by 60%. The has-news flags (and, to a lesser "
          f"extent, the averages) are therefore not stationary: disclose it, or re-run without the has-news inputs as a robustness check.")
sent_daily.head()

### 2.3 Evidence for the forecast horizon (why `HORIZON = 5`)

The horizon is both the forward-return window of the label and the rebalance interval. It is justified from the data and from prior work, and the cell below prints a warning if the configured value is poorly supported.

**Method.** A signal is evaluated by its information coefficient (Spearman rank correlation with subsequent returns) over several holding periods, together with factor turnover, because high turnover means high trading costs (Jansen, 2020). The cell does this for (i) FinBERT and Qwen sentiment averaged over the last L sessions and (ii) price-based features as a reference, and adds the trading-cost side for a top-K book. Overlapping h-day returns make neighbouring IC values dependent, so t-statistics are Newey-West (HAC) with h lags and the information ratio uses non-overlapping dates.

**Prior work.** Heston and Sinha (2017) find that news aggregated over one day predicts returns for only 1 to 2 days, while news aggregated over a week predicts them for longer, which is why the table has one row per aggregation window. Tetlock, Saar-Tsechansky and Macskassy (2008) find that prices briefly underreact to negative words in firm-specific news.

**Which row matters.** The model's sentiment features average the last L sessions, with L chosen in Section 3. The check for that window is in the cell after Section 3.

**Reading the output.** IC values are small and most cells are individually insignificant with only 7 stocks, so the defensible statements are about shape: where IC is positive, where it fades, and how trading-cost drag falls with the horizon. The choice is a judgement among horizons the data do not reject, weighed against trading costs and the number of non-overlapping test periods (fewer with longer horizons). Section 2.3b tests how much weight these tables can bear.

In [ ]:
# ---- 2.3 Evidence for the forecast horizon: rank IC by forward horizon (Alphalens style) ----------------
# For every horizon h and every sentiment aggregation window L (sessions), the signal is the average FinBERT / Qwen score of the
# last L sessions' headlines; the outcome is the h-session forward return. IC = cross-sectional Spearman correlation per date
# (7 stocks, >= 4 with a signal), averaged over dates. Overlapping h-day returns make neighbouring IC values dependent, so the
# t-statistics are Newey-West (HAC) with h lags, and the information ratio uses non-overlapping dates only.
HZ_CANDIDATES = sorted(set([1, 2, 3, 5, 10, 21, 42, int(HORIZON)]))
HZ_AGG_WINDOWS = [1, 3, 5, 10, 21]

def xs_rank_ic(sig, fwd, min_names=4):
    m = sig.notna() & fwd.notna()
    rs, rf = sig.where(m).rank(axis=1), fwd.where(m).rank(axis=1)
    rs_c, rf_c = rs.sub(rs.mean(axis=1), axis=0), rf.sub(rf.mean(axis=1), axis=0)
    den = np.sqrt((rs_c ** 2).sum(axis=1) * (rf_c ** 2).sum(axis=1))
    return ((rs_c * rf_c).sum(axis=1) / den).where((m.sum(axis=1) >= min_names) & (den > 0)).dropna()

def hac_t(x, lags):
    x = np.asarray(x, float); n = len(x); e = x - x.mean(); v = (e @ e) / n
    for k in range(1, lags + 1):
        v += 2 * (1 - k / (lags + 1)) * (e[k:] @ e[:-k]) / n
    return x.mean() / np.sqrt(v / n)

def ic_by_horizon(sig_panels, close, horizons, start=None, extra_lag=None):
    """sig_panels: {label: dates x tickers signal panel}. Returns mean IC, HAC t, annualised IC information ratio by (label, horizon)."""
    ic_, t_, ir_ = {}, {}, {}
    for lab, sig in sig_panels.items():
        for h in horizons:
            ic = xs_rank_ic(sig, close.shift(-h) / close - 1)
            if start is not None:
                ic = ic[ic.index >= start]
            non = ic.iloc[::h]
            ic_[(lab, h)], t_[(lab, h)] = ic.mean(), hac_t(ic.values, h + (extra_lag or {}).get(lab, 0))   # lag: overlapping returns + overlapping signal window
            ir_[(lab, h)] = non.mean() / non.std() * np.sqrt(252 / h) if len(non) > 10 and non.std() > 0 else np.nan
    return [pd.Series(d).unstack() for d in (ic_, t_, ir_)]

_close = eq_raw.pivot(index='date', columns='symbol', values='Close').sort_index().reindex(columns=TICKERS)
_S = {src: sent_daily.pivot(index='date', columns='ticker', values=f'{src}_sum').reindex(_close.index).reindex(columns=TICKERS).fillna(0.0)
      for src in ('finbert', 'qwen')}
_N = sent_daily.pivot(index='date', columns='ticker', values='n_headlines').reindex(_close.index).reindex(columns=TICKERS).fillna(0.0)
def _agg_panels(src):
    out = {}
    for L in HZ_AGG_WINDOWS:
        n = _N.rolling(L, min_periods=1).sum()
        out[L] = (_S[src].rolling(L, min_periods=1).sum() / n).where(n > 0)
    return out

_sent_start = pd.Timestamp(SENTIMENT_START)
hz_ic, hz_t, hz_ir = {}, {}, {}
for src, label in (('finbert', 'FinBERT'), ('qwen', 'Qwen')):
    hz_ic[label], hz_t[label], hz_ir[label] = ic_by_horizon(_agg_panels(src), _close, HZ_CANDIDATES, start=_sent_start, extra_lag={L: L - 1 for L in HZ_AGG_WINDOWS})
    print(f"\n===== {label}: mean rank IC by sentiment aggregation window (rows, sessions) and forward horizon (columns, sessions) =====")
    print(hz_ic[label].round(4).to_string())
    print(f"\n{label}: Newey-West t-statistics"); print(hz_t[label].round(2).to_string())
    print(f"\n{label}: annualised IC information ratio (non-overlapping dates)"); print(hz_ir[label].round(2).to_string())

_price_sig = {'1-day return (reversal)': _close / _close.shift(1) - 1, '5-day return': _close / _close.shift(5) - 1,
              '21-day return': _close / _close.shift(21) - 1, '12-1 month momentum': _close.shift(21) / _close.shift(252) - 1}
hz_ic['price'], hz_t['price'], _ = ic_by_horizon(_price_sig, _close, HZ_CANDIDATES, start=_close.index[300],
                                                  extra_lag={'1-day return (reversal)': 0, '5-day return': 4, '21-day return': 20, '12-1 month momentum': 20})
print("\n===== Price-based features (same method, full price history): mean rank IC by forward horizon =====")
print(hz_ic['price'].round(4).to_string())

# trading-cost side: turnover of a top-3 book driven by the 5-session FinBERT signal, rebalanced every h sessions
_sig5 = _agg_panels('finbert')[5].loc[_sent_start:]
_cost_rows = []
for h in HZ_CANDIDATES:
    prev, tos = None, []
    for _, r in _sig5.iloc[::h].dropna(thresh=4).iterrows():
        top = set(r.dropna().nlargest(PORTFOLIO_TOP_K).index)
        if prev is not None:
            tos.append(1 - len(top & prev) / PORTFOLIO_TOP_K)
        prev = top
    to = float(np.mean(tos)) if tos else np.nan
    _cost_rows.append(dict(horizon=h, rebalances_per_year=252 / h, turnover_per_rebalance=to,
                           cost_drag_pct_per_year=to * (252 / h) * 2 * TRANSACTION_COST_BPS / 1e4 * 100))
hz_cost = pd.DataFrame(_cost_rows).set_index('horizon')
print(f"\n===== Trading-cost side (top-{PORTFOLIO_TOP_K} book on the 5-session FinBERT signal, {TRANSACTION_COST_BPS} bp one-way) =====")
print(hz_cost.round(2).to_string())

# ---- summary: best horizon for every sentiment aggregation window. Which window the MODEL uses is only known after the AR/BIC step
# (Section 3), so the check against the configured horizon for that window is run there.
print(f"\n===== Best horizon by sentiment aggregation window (configured HORIZON = {HORIZON}) =====")
for label in ('FinBERT', 'Qwen'):
    for L in HZ_AGG_WINDOWS:
        row = hz_ic[label].loc[L]; best = int(row.idxmax()); mine = float(row.get(HORIZON, np.nan))
        share = f"{mine / row.max():.0%} of its peak" if (row.max() > 0 and mine > 0) else "not positive"
        print(f"  {label:8s} {L:>2}-session average: IC peaks at h={best:>2} ({row.max():+.4f}); at h={HORIZON}: {mine:+.4f} ({share})")
_ntest = sum(hz_t[s_].size for s_ in ('FinBERT', 'Qwen')); _nsig = sum(int((hz_t[s_].abs() > 2).sum().sum()) for s_ in ('FinBERT', 'Qwen'))
print(f"{_nsig} of {_ntest} (aggregation window, horizon) cells have |Newey-West t| > 2. Longer averaging windows tend to move the best horizon "
      f"outward, so the horizon has to be read together with the sentiment window the model uses (checked after Section 3). Where few cells are "
      f"individually significant the data cannot single out one best horizon: use the trading-cost drag and the number of non-overlapping periods to break the tie.")

fig, axes = plt.subplots(1, 3, figsize=(17, 4.3))
for ax, label in zip(axes[:2], ('FinBERT', 'Qwen')):
    for L in HZ_AGG_WINDOWS:
        ax.plot(HZ_CANDIDATES, hz_ic[label].loc[L].values, marker='o', lw=1.2, label=f'{L}-session average')
    ax.axhline(0, color='grey', lw=1); ax.axvline(HORIZON, color='black', ls='--', lw=1)
    ax.set_xscale('log'); ax.set_xticks(HZ_CANDIDATES); ax.set_xticklabels(HZ_CANDIDATES)
    ax.set_title(f'{label} sentiment: mean rank IC vs forward horizon'); ax.set_xlabel('forward horizon (sessions)'); ax.legend(fontsize=7)
for name in hz_ic['price'].index:
    axes[2].plot(HZ_CANDIDATES, hz_ic['price'].loc[name].values, marker='o', lw=1.2, label=name)
axes[2].axhline(0, color='grey', lw=1); axes[2].axvline(HORIZON, color='black', ls='--', lw=1)
axes[2].set_xscale('log'); axes[2].set_xticks(HZ_CANDIDATES); axes[2].set_xticklabels(HZ_CANDIDATES)
axes[2].set_title('Price-based features: mean rank IC vs forward horizon'); axes[2].set_xlabel('forward horizon (sessions)'); axes[2].legend(fontsize=7)
plt.tight_layout(); plt.savefig('horizon_evidence.png', dpi=120, bbox_inches='tight'); plt.show()

pd.concat({f'{k}_IC': hz_ic[k] for k in ('FinBERT', 'Qwen', 'price')} | {f'{k}_t': hz_t[k] for k in ('FinBERT', 'Qwen', 'price')}).to_csv('horizon_evidence_ic.csv')
hz_cost.to_csv('horizon_evidence_cost.csv')

### 2.3b How much weight can the IC tables bear?

The tables above search 35 (averaging window, horizon) cells per sentiment source, with only 7 stocks. A peak found by searching is expected even when nothing predicts returns, so it is only informative if it would rarely arise by chance. The cell below:

1. computes the **family-wise p-value** of the best cell by circularly shifting each stock's sentiment series (this breaks any link with returns but keeps the series' own persistence) and recomputing the whole grid,
2. reports how often a **single** cell shows |t| > 2 when there is no relationship (nominally 5%),
3. computes the **partial IC** after removing what trailing 5- and 21-session returns explain, because sentiment is correlated with recent returns and may only re-measure momentum,
4. compares the **two halves** of the sample.

Newey-West lags are h + L - 1, because overlapping forward returns and overlapping sentiment windows both create autocorrelation in the IC series. If the family-wise p-value is large, the data do not single out a horizon, and the horizon has to be justified on other grounds (trading cost, number of non-overlapping test periods, prior work).

In [ ]:
# ---- 2.3b Reliability checks for the horizon evidence ---------------------------------------------------------
# Three questions the IC tables above cannot answer by themselves:
#  (1) SEARCH: with 35 (window, horizon) cells per source, how good a PEAK does noise alone produce? Each stock's sentiment series is
#      circularly shifted by a random number of days (breaking any link to returns, keeping its own autocorrelation) and the whole
#      grid is recomputed; the family-wise p-value is the share of shifted grids whose best |t| (or best IC) is at least as large.
#  (2) SIZE: how often does a single cell show |t| > 2 when there is NO relationship? (nominal: 5%)
#  (3) CONFOUNDING and STABILITY: is the IC just price momentum/reversal (partial IC after removing the cross-sectional ranks of the
#      trailing 5- and 21-session returns), and does it hold in both halves of the sample?
HZ_NPERM = 100
_rng = np.random.default_rng(RANDOM_SEED)
_win = _close.index[_close.index >= _sent_start]
_FW = {h: (_close.shift(-h) / _close - 1).reindex(_win) for h in HZ_CANDIDATES}
_SRC = {'FinBERT': 'finbert', 'Qwen': 'qwen'}

def _grid(Ssum, Ncnt):
    ic, t = {}, {}
    for L in HZ_AGG_WINDOWS:
        n = Ncnt.rolling(L, min_periods=1).sum(); sig = (Ssum.rolling(L, min_periods=1).sum() / n).where(n > 0)
        for h in HZ_CANDIDATES:
            x = xs_rank_ic(sig, _FW[h]); ic[(L, h)] = x.mean(); t[(L, h)] = hac_t(x.values, h + L - 1)   # lag covers BOTH overlaps
    return pd.Series(ic).unstack(), pd.Series(t).unstack()

def _signal(label, L):
    n = _N.reindex(_win).rolling(L, min_periods=1).sum()
    return (_S[_SRC[label]].reindex(_win).rolling(L, min_periods=1).sum() / n).where(n > 0)

_ctrl = [(_close / _close.shift(k) - 1).reindex(_win).rank(axis=1) for k in (5, 21)]
def partial_ic_cell(label, L, h):
    """Rank IC of the sentiment signal after removing, date by date, what trailing 5- and 21-session return ranks explain."""
    rs, rf = _signal(label, L).rank(axis=1), _FW[h].rank(axis=1); ics = []
    for dt in _win:
        y, f = rs.loc[dt], rf.loc[dt]; X = pd.concat([c.loc[dt] for c in _ctrl], axis=1)
        m = y.notna() & f.notna() & X.notna().all(axis=1)
        if m.sum() < 5: continue
        A = np.column_stack([np.ones(int(m.sum()))] + [X[m].iloc[:, j].values for j in range(X.shape[1])])
        resid = y[m].values - A @ np.linalg.lstsq(A, y[m].values, rcond=None)[0]
        if resid.std() > 0: ics.append(pd.Series(resid).rank().corr(f[m].reset_index(drop=True).rank()))
    ics = np.array([v for v in ics if v == v]); return ics.mean(), hac_t(ics, h + L - 1)

def subperiod_cell(label, L, h):
    x = xs_rank_ic(_signal(label, L), _FW[h]); mid = _win[len(_win) // 2]
    return x[x.index < mid].mean(), x[x.index >= mid].mean(), mid

hz_rel = []
for label in ('FinBERT', 'Qwen'):
    S_, N_ = _S[_SRC[label]].reindex(_win), _N.reindex(_win); T_ = len(S_)
    ic0, t0 = _grid(S_, N_); obs_t, obs_ic = t0.abs().max().max(), ic0.max().max()
    nt, ni, allt = [], [], []
    for _ in range(HZ_NPERM):
        Ss, Ns = S_.copy(), N_.copy()
        for c in S_.columns:
            k = int(_rng.integers(40, T_ - 40)); Ss[c] = np.roll(S_[c].values, k); Ns[c] = np.roll(N_[c].values, k)
        icb, tb = _grid(Ss, Ns); nt.append(tb.abs().max().max()); ni.append(icb.max().max()); allt.append(tb.abs().values.ravel())
    nt, ni = np.array(nt), np.array(ni)
    pL, ph = [int(v) for v in ic0.stack().idxmax()]
    pic, pt = partial_ic_cell(label, pL, ph); a, b, mid = subperiod_cell(label, pL, ph)
    hz_rel.append(dict(source=label, peak_window=pL, peak_horizon=ph, peak_IC=ic0.loc[pL, ph], max_abs_t=obs_t,
                       null_max_abs_t_median=np.median(nt), null_max_abs_t_95pct=np.percentile(nt, 95),
                       familywise_p_maxt=(np.sum(nt >= obs_t) + 1) / (HZ_NPERM + 1), familywise_p_peakIC=(np.sum(ni >= obs_ic) + 1) / (HZ_NPERM + 1),
                       false_positive_rate_abs_t_gt_2=float(np.mean(np.concatenate(allt) > 2)),
                       partial_IC=pic, partial_t=pt, IC_first_half=a, IC_second_half=b))
hz_rel = pd.DataFrame(hz_rel).set_index('source')
hz_rel.to_csv('horizon_evidence_reliability.csv')
print(f"Reliability of the horizon evidence ({HZ_NPERM} circular-shift permutations per source; sentiment window {_win[0].date()} .. {_win[-1].date()}):")
for label, r in hz_rel.iterrows():
    print(f"\n{label}: strongest cell = {int(r.peak_window)}-session average at h={int(r.peak_horizon)} (IC {r.peak_IC:+.4f}, max |t| over the grid {r.max_abs_t:.2f})")
    print(f"  (1) search: under no relationship the best |t| over the grid has median {r.null_max_abs_t_median:.2f} and 95th percentile {r.null_max_abs_t_95pct:.2f}; "
          f"family-wise p = {r.familywise_p_maxt:.2f} (best IC: p = {r.familywise_p_peakIC:.2f})")
    print(f"  (2) size: a single cell shows |t| > 2 in {r.false_positive_rate_abs_t_gt_2:.1%} of null grids (nominal 5%)" + (" -> the t-statistics overstate significance" if r.false_positive_rate_abs_t_gt_2 > 0.07 else ""))
    print(f"  (3) partial IC after removing trailing-return ranks: {r.partial_IC:+.4f} (t = {r.partial_t:.2f}) vs plain {r.peak_IC:+.4f}; IC in first half {r.IC_first_half:+.4f}, second half {r.IC_second_half:+.4f}")
    if r.familywise_p_maxt > 0.10:
        print(f"  => The strongest cell is NOT distinguishable from what searching noise produces (family-wise p > 0.10): do not present the peak horizon as established.")

## 3. Stationarity and data-driven look-back periods

Each variable's look-back window is estimated from the data instead of being fixed by hand. **Every estimate uses only data dated before the first out-of-sample date** (`FIRST_OOS_DATE`), and the sentiment look-backs use the first year of headline data only (Section 2.2). The look-backs are therefore fixed before any test data is seen.

**Step 1: integration order *d* (the "I" in ARIMA).** For each raw series, the augmented Dickey-Fuller (ADF) test (null: unit root) decides whether it must be differenced. KPSS (null: stationary) is reported alongside as a cross-check. The smallest *d* for which ADF rejects at 5% is used. Prices, volumes and most yields are I(1) and enter the models as changes or returns. The VIX and spreads are typically I(0) and enter as levels.

**Step 2: look-back from an autoregression.** On the stationary version of each series, an AR(p) is fitted for p = 0 to 20 on a common estimation sample, and BIC picks *p* (the number of past days with direct predictive content). A persistent series keeps "remembering" a shock for longer than its AR order suggests (an AR(1) with coefficient 0.95 has order 1 but a long memory), so the look-back is

$$L = \max\left(p,\; \min\{k : |\psi_j| < 0.10 \;\; \forall j \ge k\}\right)$$

where $\psi_j$ is the impulse response of the fitted AR(p). L is the number of days until a shock's effect has decayed below 10% of its initial size. Slow state variables (macro, VIX, market return) get $L \ge$ `HORIZON`, so that a feature never aggregates less than the label horizon (reasons in 3.0 below).

**Step 3: VIX regimes.** The same AR/BIC selection is repeated separately on low-VIX and high-VIX days (split at the selection window's median VIX). If memory differs by regime, the feature uses the regime-specific window on each date.

**Step 4: ARIMA feature.** For each stock, an ARIMA(p,0,q) on daily log returns (d = 0 from Step 1) with p, q <= 2 is chosen by BIC. It is re-estimated each January on all prior data and then run forward with fixed parameters, giving a one-step-ahead forecast and today's innovation as features.

### 3.0 Why these rules, and where they come from

The look-back procedure combines standard, literature-based tools in a way that is my own design: an AR/BIC plus impulse-response rule built on Box-Jenkins tools.

**Look-back rule: $L = \max(p, \text{impulse-response decay})$**

| Component | Status | Reason | Reference |
|---|---|---|---|
| Test stationarity before modelling, difference until stationary | Literature | Regressions on non-stationary series give spurious results, and ARIMA's *d* is defined this way | Box & Jenkins (1970), Dickey & Fuller (1979), Kwiatkowski et al. (1992) |
| Choose the AR order *p* by BIC | Literature | BIC is consistent for the true lag order and penalises extra lags more than AIC, which suits noisy daily financial data | Schwarz (1978), Hamilton (1994) |
| Measure memory with the impulse response | Literature (standard tool) | *p* counts lags with direct effects, but a persistent series (for example an AR(1) with coefficient 0.95) carries a shock for many more days. The impulse response measures that total persistence. The trading version of the same idea is the half-life of mean reversion. | Hamilton (1994), Chan (2013) |
| Combine them as max(p, decay) | **Own design** | Neither piece alone works for all variables. For near-white-noise returns the decay is 0 to 1 days, so *p* matters. For persistent series such as the VIX, *p* = 1 hides a long memory, so the decay matters. | None |
| 10% decay tolerance (`IRF_TOL`) | **Own design (convention)** | Similar to a half-life, which uses 50%, but stricter, so the window covers most of a shock's effect. It is a convention and is not estimated. | None |
| Cap of 126 days (`MAX_LOOKBACK`) | **Own design** | AR models estimated on daily data are unreliable about effects more than about 6 months out | None |
| Floor of `HORIZON` days for slow variables | **Own design** | A feature that averages fewer days than the 5-day label is mostly noise relative to what it has to predict. The floor aligns feature and label horizons. | None |

**VIX-regime look-backs**

| Component | Status | Reason | Reference |
|---|---|---|---|
| Return dynamics differ between calm and volatile markets | Literature | Serial correlation of index returns falls when volatility is high, the predictive power of news sentiment is concentrated in bad times, and return processes are better described as regime-switching than as one stable process | LeBaron (1992), Garcia (2013), Hamilton (1989), Ang & Bekaert (2002) |
| Two regimes, split at the median VIX of the selection window, with the same selection rule in each | **Own design** | Simple and fixed in advance. It uses an observable variable, so there is no latent state to estimate, and smoothed hidden-Markov regime probabilities would leak future information. The median split gives each regime the same sample size. A Markov-switching AR (Hamilton, 1989) is the standard alternative. | None |

**Position scaling**

| Component | Status | Reason | Reference |
|---|---|---|---|
| Inverse-volatility weights within the portfolio | Literature | Equalises each name's risk contribution when correlations are ignored ("naive risk parity") | Maillard, Roncalli & Teiletche (2010), Jansen (2020) |
| Reduce exposure when volatility is high | Literature | Volatility-managed portfolios improve Sharpe because volatility is persistent while expected returns do not rise proportionally, and momentum-type strategies crash mainly in high-volatility periods | Moreira & Muir (2017), Barroso & Santa-Clara (2015), Daniel & Moskowitz (2016), Harvey et al. (2018) |
| VIX ranked by its own expanding percentile, with a linear cut from the 70th to the 95th percentile down to 30% exposure | **Own design** | The VIX is forward-looking implied volatility, available daily with no estimation. A percentile adapts to the 2010 to 2025 distribution using past data only. The 70/95/30% values were fixed in advance and not optimised. The literature more often targets realised volatility (Moreira & Muir), which is the natural alternative. | None |
| Drawdown circuit breaker (15% out, 7% back in) | **Practitioner rule** | Risk control, not a source of return. No specific paper. | None |

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tools.sm_exceptions import InterpolationWarning

_td = pd.DatetimeIndex(trading_dates)
FIRST_OOS_DATE = _td[_td >= pd.Timestamp(MODEL_START) + pd.DateOffset(years=INITIAL_TRAIN_YEARS)][0]
print(f"First out-of-sample date: {FIRST_OOS_DATE.date()}  (all look-back selection uses data before it)")

# --- raw daily series --------------------------------------------------------
panel = eq_raw.rename(columns={'symbol': 'ticker'}).sort_values(['ticker', 'date']).reset_index(drop=True)
panel['logret'] = panel.groupby('ticker')['Close'].transform(lambda c: np.log(c).diff())

mk = mkt_raw.pivot(index='date', columns='symbol', values='Close').rename(
    columns={'^GSPC': 'sp500', '^VIX': 'vix', '^VIX3M': 'vix3m'}).reindex(_td).ffill(limit=5)
fred_lag = fred_raw.copy()
fred_lag.index = fred_lag.index + pd.offsets.BDay(1)        # 1-business-day publication lag
fred_lag = fred_lag[~fred_lag.index.duplicated(keep='last')]
fred_lag = fred_lag.reindex(fred_lag.index.union(_td)).sort_index().ffill(limit=5).reindex(_td)
market = mk.join(fred_lag)
market['mkt_logret'] = np.log(market['sp500']).diff()
market['log_vix'] = np.log(market['vix'])
market['log_usd_broad'] = np.log(market['usd_broad'])
market['log_brent'] = np.log(market['brent'])
market['term_spread'] = market['ust10y'] - market['tbill3m']
market.index.name = 'date'

# daily sentiment level per ticker (0 on no-news days), on the trading calendar
sent_grid = (pd.MultiIndex.from_product([TICKERS, _td[_td >= pd.Timestamp(SENTIMENT_START)]], names=['ticker', 'date'])
             .to_frame(index=False).merge(sent_daily, on=['ticker', 'date'], how='left').fillna(0.0))
for src in ['finbert', 'qwen']:
    sent_grid[f'{src}_daily'] = np.where(sent_grid['n_headlines'] > 0,
                                         sent_grid[f'{src}_sum'] / sent_grid['n_headlines'].clip(lower=1), 0.0)

pre_oos = lambda idx: idx < FIRST_OOS_DATE
sent_sel = lambda idx: (idx >= pd.Timestamp(SENTIMENT_START)) & (idx < pd.Timestamp(SENTIMENT_SELECTION_END))

In [ ]:
def stationarity_tests(x):
    x = pd.Series(x).dropna().values
    adf_p = adfuller(x, autolag='AIC')[1]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', InterpolationWarning)
        kpss_p = kpss(x, regression='c', nlags='auto')[1]
    return adf_p, kpss_p

def diff_n(x, d):
    """d-th difference: the first difference applied d times. NOT x.diff(d), which is the lag-d difference x_t - x_{t-d}
    (for d = 2 that is not stationary for an I(2) series, whereas x.diff().diff() is)."""
    for _ in range(d):
        x = x.diff()
    return x

def integration_order(x, max_d=2, alpha=0.05):
    """Smallest d at which ADF rejects a unit root (ARIMA's differencing order).
    KPSS p-value at that d is returned as a cross-check (KPSS > alpha = agreement)."""
    x = pd.Series(x).dropna()
    for d in range(max_d + 1):
        xd = diff_n(x, d).dropna()
        adf_p, kpss_p = stationarity_tests(xd)
        if adf_p < alpha:
            return d, adf_p, kpss_p
    return max_d, adf_p, kpss_p

# Unit test: series that need 0, 1 and 2 differences must give d = 0, 1, 2.
_g = np.random.default_rng
assert integration_order(pd.Series(_g(2).normal(size=1500)))[0] == 0
assert integration_order(pd.Series(np.cumsum(_g(1).normal(size=1500))))[0] == 1
assert integration_order(pd.Series(np.cumsum(np.cumsum(_g(0).normal(size=1500)))))[0] == 2, "integration_order misclassifies an I(2) series"
print("Differencing unit test passed: I(0), I(1) and I(2) series are classified d = 0, 1, 2.")

# (name, list-of-series, scope) -- per-ticker series are tested ticker by ticker, d = max over tickers
def _per_ticker(col, mask_fn, frame=panel, transform=None):
    out = []
    for t, g in frame.groupby('ticker'):
        s = g.set_index('date')[col]
        s = s[mask_fn(s.index)]
        out.append(transform(s) if transform else s)
    return out

STATIONARITY_CANDIDATES = {
    'stock_log_price': (_per_ticker('Close', pre_oos, transform=np.log), 'pre-OOS'),
    'stock_return': (_per_ticker('logret', pre_oos), 'pre-OOS'),
    'stock_log_volume': (_per_ticker('Volume', pre_oos, transform=lambda s: np.log(s.replace(0, np.nan))), 'pre-OOS'),
    'sp500_log_level': ([np.log(market.loc[pre_oos(market.index), 'sp500'])], 'pre-OOS'),
    'sp500_return': ([market.loc[pre_oos(market.index), 'mkt_logret']], 'pre-OOS'),
    'log_vix': ([market.loc[pre_oos(market.index), 'log_vix']], 'pre-OOS'),
    'ust10y': ([market.loc[pre_oos(market.index), 'ust10y']], 'pre-OOS'),
    'term_spread': ([market.loc[pre_oos(market.index), 'term_spread']], 'pre-OOS'),
    'credit_spread': ([market.loc[pre_oos(market.index), 'credit_spread']], 'pre-OOS'),
    'breakeven10y': ([market.loc[pre_oos(market.index), 'breakeven10y']], 'pre-OOS'),
    'log_usd_broad': ([market.loc[pre_oos(market.index), 'log_usd_broad']], 'pre-OOS'),
    'log_brent': ([market.loc[pre_oos(market.index), 'log_brent']], 'pre-OOS'),
    'finbert_daily': (_per_ticker('finbert_daily', sent_sel, frame=sent_grid), SENTIMENT_START[:4]),
    'qwen_daily': (_per_ticker('qwen_daily', sent_sel, frame=sent_grid), SENTIMENT_START[:4]),
}

stat_rows, INTEGRATION_ORDER = [], {}
for name, (series_list, window) in STATIONARITY_CANDIDATES.items():
    res = [integration_order(s) for s in series_list if s.dropna().nunique() > 5]
    if not res:     # e.g. almost no headlines in the selection year
        print(f"WARNING: {name}: too few distinct values in the selection window to test; assuming d=0")
        res = [(0, np.nan, np.nan)]
    d = max(r[0] for r in res)
    INTEGRATION_ORDER[name] = d
    stat_rows.append(dict(series=name, window=window, n_series=len(res), d=d,
                          adf_p_at_d_median=np.median([r[1] for r in res]),
                          kpss_p_at_d_median=np.median([r[2] for r in res]),
                          kpss_agrees=all(r[2] > 0.05 for r in res)))
stationarity_df = pd.DataFrame(stat_rows)
print("Integration order d (ADF decides; KPSS > 0.05 = KPSS agrees the transformed series is stationary):")
print(stationarity_df.round(4).to_string(index=False))
stationarity_df.to_csv('stationarity_tests.csv', index=False)

In [ ]:
def _lag_matrix(x, maxlag):
    n = len(x)
    X = np.column_stack([x[maxlag - k:n - k] for k in range(1, maxlag + 1)])
    return x[maxlag:], X

def select_ar_order_bic(series_list, maxlag=MAX_AR_LAG, row_masks=None, min_rows=100):
    """Pooled OLS AR(p), p = 0..maxlag, all on the SAME estimation sample so BIC values are
    comparable across p. Lags never cross tickers (lag matrices built per series, then stacked).
    row_masks (optional): per-series boolean arrays choosing which target days enter
    (used for regime-conditional selection; the lags themselves still come from all days)."""
    Ys, Xs = [], []
    for i, s in enumerate(series_list):
        x = np.asarray(pd.Series(s).ffill().dropna(), float)
        if len(x) <= maxlag + 10:
            continue
        y, X = _lag_matrix(x, maxlag)
        if row_masks is not None:
            m = np.asarray(row_masks[i], bool)[-len(y):]
            y, X = y[m], X[m]
        Ys.append(y); Xs.append(X)
    y = np.concatenate(Ys); X = np.vstack(Xs); n = len(y)
    if n < min_rows:
        return None
    bics, betas = [], []
    for p in range(maxlag + 1):
        Z = np.column_stack([np.ones(n), X[:, :p]])
        beta = np.linalg.lstsq(Z, y, rcond=None)[0]
        rss = float(((y - Z @ beta) ** 2).sum())
        bics.append(n * np.log(rss / n) + (p + 1) * np.log(n))
        betas.append(beta)
    p = int(np.argmin(bics))
    return dict(p=p, phi=betas[p][1:], bic=bics, n=n)

def memory_length(p, phi, tol=IRF_TOL, max_len=MAX_LOOKBACK):
    """Look-back L = max(p, days until the AR(p) impulse response stays below tol)."""
    if p == 0:
        return 1
    H = max_len + 60
    psi = np.zeros(H); psi[0] = 1.0
    for j in range(1, H):
        psi[j] = sum(phi[k] * psi[j - 1 - k] for k in range(min(p, j)))
    above = np.where(np.abs(psi[1:]) >= tol)[0]
    k = int(above.max() + 1) if len(above) else 1
    return int(np.clip(max(p, k), 1, max_len))

def _stationary(s, d):
    s = pd.Series(s)
    return diff_n(s, d).dropna()

def _vix_on(index):
    return market['vix'].reindex(index).ffill().values

def select_lookback(name, series_list, d, min_window=1):
    stat_list = [_stationary(s, d) for s in series_list]
    base = select_ar_order_bic(stat_list)
    L = max(memory_length(base['p'], base['phi']), min_window)
    # regime-conditional: split target days by the selection window's median VIX
    vix_all = np.concatenate([_vix_on(s.index) for s in stat_list])
    thr = float(np.nanquantile(vix_all, VIX_REGIME_QUANTILE))
    out = dict(variable=name, d=d, p=base['p'], L=L, n=base['n'], vix_threshold=thr)
    for regime, fn in [('low', lambda v: v < thr), ('high', lambda v: v >= thr)]:
        masks = [fn(_vix_on(s.index)) for s in stat_list]
        r = select_ar_order_bic(stat_list, row_masks=masks)
        if r is None:
            out[f'p_{regime}'], out[f'L_{regime}'] = base['p'], L       # too few days: fall back
        else:
            out[f'p_{regime}'] = r['p']
            out[f'L_{regime}'] = max(memory_length(r['p'], r['phi']), min_window)
    return out

LB_CANDIDATES = {   # variable: (source series key, minimum window)
    'stock_return': ('stock_return', 1),
    'sp500_return': ('sp500_return', HORIZON),
    'log_vix': ('log_vix', HORIZON),
    'ust10y': ('ust10y', HORIZON),
    'term_spread': ('term_spread', HORIZON),
    'credit_spread': ('credit_spread', HORIZON),
    'breakeven10y': ('breakeven10y', HORIZON),
    'log_usd_broad': ('log_usd_broad', HORIZON),
    'log_brent': ('log_brent', HORIZON),
    'finbert_daily': ('finbert_daily', 1),
    'qwen_daily': ('qwen_daily', 1),
}
lb_rows = []
for var, (key, min_w) in LB_CANDIDATES.items():
    series_list, _ = STATIONARITY_CANDIDATES[key]
    lb_rows.append(select_lookback(var, series_list, INTEGRATION_ORDER[key], min_window=min_w))
lookback_df = pd.DataFrame(lb_rows)[['variable', 'd', 'p', 'L', 'p_low', 'L_low', 'p_high', 'L_high', 'vix_threshold', 'n']]
LOOKBACKS = {r['variable']: r for r in lb_rows}
P_RET = max(LOOKBACKS['stock_return']['p'], 1)   # AR lags of the stock's own return used as features
print("Data-driven look-backs (trading days). p = BIC-selected AR order on the stationary series;")
print("L = max(p, impulse-response decay below 10%); low/high = the same selection within VIX regimes.")
print(lookback_df.round(2).to_string(index=False))
lookback_df.to_csv('lookback_selection.csv', index=False)

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

def _fit_arima(y, order):
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter('always')
        res = ARIMA(y, order=order, trend='c').fit()
    converged = bool((res.mle_retvals or {}).get('converged', True))
    return res, converged

def select_arima_order(y, max_p=ARIMA_MAX_P, max_q=ARIMA_MAX_Q):
    """BIC over ARIMA(p,0,q), excluding (0,0,0) (a constant forecast is not a usable feature).
    Non-converged fits are skipped -- their BIC is not a maximum."""
    best, rows = (np.inf, (1, 0, 0)), []
    for p in range(max_p + 1):
        for q in range(max_q + 1):
            if p == 0 and q == 0:
                continue
            try:
                res, conv = _fit_arima(y, (p, 0, q))
                rows.append(dict(order=(p, 0, q), bic=res.bic, converged=conv))
                if conv and res.bic < best[0]:
                    best = (res.bic, (p, 0, q))
            except Exception:
                pass
    return best[1], pd.DataFrame(rows)

ARIMA_ORDERS, arima_rows = {}, []
for t, g in panel.groupby('ticker'):
    y = (g.set_index('date')['logret'] * 100).dropna()
    y = y[y.index < FIRST_OOS_DATE].values
    order, grid = select_arima_order(y)
    ARIMA_ORDERS[t] = order
    arima_rows.append(dict(ticker=t, order=order, n=len(y),
                           bic=grid.loc[grid['order'] == order, 'bic'].min()))
arima_order_df = pd.DataFrame(arima_rows)
print("ARIMA(p, d=0, q) order per stock, selected by BIC on pre-OOS daily log returns:")
print(arima_order_df.to_string(index=False))

# LSTM sequence length: the longest selected memory among the fast, stock-level inputs.
# Slow state variables (VIX, macro, fundamentals) already enter each time step as levels/z-scores
# that summarise their own long windows, so they do not need to be re-read inside the sequence.
_fast = ['stock_return', 'finbert_daily', 'qwen_daily']
_mem = max(max(LOOKBACKS[v]['L'], LOOKBACKS[v]['L_low'], LOOKBACKS[v]['L_high']) for v in _fast)
SEQ_LEN = int(np.clip(max(_mem, P_RET), *LSTM_SEQ_BOUNDS))
print(f"\nLSTM sequence length = {SEQ_LEN} (longest fast-variable memory = {_mem}, bounds {LSTM_SEQ_BOUNDS}); v12 used a fixed 10.")
if max(_mem, P_RET) < LSTM_SEQ_BOUNDS[0]:
    print(f"NOTE: the data-driven memory estimate ({max(_mem, P_RET)}d) is below the lower bound ({LSTM_SEQ_BOUNDS[0]}d), "
          f"so the bound -- a hard-coded guardrail -- and not the AR/BIC estimate sets SEQ_LEN = {SEQ_LEN}. State this in the write-up.")

fig, ax = plt.subplots(figsize=(10, 4))
plot_df = lookback_df.set_index('variable')[['L', 'L_low', 'L_high']]
plot_df.plot(kind='bar', ax=ax)
ax.axhline(HORIZON, color='grey', ls='--', lw=1, label=f'v12 fixed window ({HORIZON}d)')
ax.set_ylabel('look-back (trading days)'); ax.set_title('Data-driven look-back per variable, overall and by VIX regime')
ax.legend(fontsize=8); plt.xticks(rotation=35, ha='right'); plt.tight_layout()
plt.savefig('lookback_selection.png', dpi=120, bbox_inches='tight'); plt.show()

### 3.1 Horizon check for the sentiment window the model actually uses

Section 2.3 evaluated several sentiment averaging windows. The model's own window is the look-back `L` selected above (`LOOKBACKS`). This cell reads the IC-by-horizon result for that window and compares it with the configured `HORIZON`.

In [ ]:
for key, label in (('finbert_daily', 'FinBERT'), ('qwen_daily', 'Qwen')):
    L_model = int(LOOKBACKS[key]['L']); L_eval = min(HZ_AGG_WINDOWS, key=lambda x: abs(x - L_model))
    row, tt = hz_ic[label].loc[L_eval], hz_t[label].loc[L_eval]
    best = int(row.idxmax()); mine = float(row.get(HORIZON, np.nan)); t_mine = float(tt.get(HORIZON, np.nan))
    share = f"{mine / row.max():.0%} of its peak" if (row.max() > 0 and mine > 0) else "not positive"
    where = f" (evaluated at the nearest tested window, {L_eval})" if L_eval != L_model else ""
    print(f"{label}: the model averages L = {L_model} session(s){where}. IC peaks at h={best} ({row.max():+.4f}); "
          f"at HORIZON={HORIZON}: {mine:+.4f}, Newey-West t = {t_mine:.2f} ({share}).")
    if not (mine > 0) or mine < 0.5 * row.max():
        print(f"  WARNING: for the window the model uses, the configured horizon has IC <= 50% of the best horizon (or is not positive): reconsider HORIZON or explain why.")
    elif abs(t_mine) < 2:
        print(f"  Note: the IC at HORIZON={HORIZON} is not individually significant (|t| < 2); say so, and rely on the shape across horizons, the cost drag and the number of test periods.")
    _pi = partial_ic_cell(label, L_eval, HORIZON); _sp = subperiod_cell(label, L_eval, HORIZON)
    print(f"  Reliability of this cell: partial IC after removing trailing-return ranks {_pi[0]:+.4f} (t = {_pi[1]:.2f}); IC in the first half {_sp[0]:+.4f}, second half {_sp[1]:+.4f}.")

## 4. Feature engineering

Feature families (the full list is built in Section 5):

| Family | Features | Look-back source |
|---|---|---|
| Price momentum / reversal | 1/5/10/21/63-day returns, 12-1 momentum, cross-sectionally demeaned 5d/21d/12-1 returns | standard horizons |
| Autoregressive (Section 3) | own-return lags 1..p, regime-adaptive mean return over L, ARIMA one-step forecast and innovation | AR/BIC, ARIMA/BIC |
| Trend | price / SMA(10, 20, 50, 200) - 1, 50/200-day MA crossover, distance to 52-week high/low, MACD and histogram scaled by price, RSI(14) | standard indicator definitions |
| Risk / liquidity | 21d and 63d volatility and their ratio, 252d beta to S&P 500, 63d idiosyncratic vol, log volume ratio, Amihud illiquidity | standard definitions |
| Market (S&P 500) | 1-day and L-day index return, 21/63-day returns, 200-day trend, 21d realised vol, stock relative strength | AR/BIC for L |
| VIX | log VIX (or its change if I(1)), z-score over L, 5-day change, VIX/VIX3M term structure, variance risk premium, expanding percentile | AR/BIC + IRF for L |
| Macro (FRED, lagged 1 day) | 10Y yield, term spread, credit spread, 10Y breakeven inflation, broad dollar, Brent -- level if I(0), L-day change if I(1) | ADF for d, AR/BIC for L |
| Fundamentals (SEC XBRL, point-in-time) | earnings yield, EPS growth, revenue growth, ROE, net margin, SUE (all cross-sectional ranks), log days since filing, availability flag | quarterly, as filed |
| Sentiment (FinBERT **or** Qwen) | average score over the source's own L (regime-adaptive), has-news flag, sentiment-available flag | AR/BIC on the first sentiment year |

**Missing values (following Gu, Kelly and Xiu, 2020):** fundamentals are converted to cross-sectional ranks in [-0.5, 0.5] on each date, and a missing value is set to 0 (the cross-sectional median) with an explicit `fund_available` flag -- this keeps META's early quarters and the 2010-2011 XBRL gaps in the panel instead of dropping rows. Sentiment before `SENTIMENT_START` is 0 with `sentiment_available = 0`. All other features require their full warm-up window; rows before it are dropped.

In [ ]:
def _rsi(c, n=14):
    d = c.diff()
    up = d.clip(lower=0).ewm(alpha=1 / n, adjust=False).mean()
    dn = (-d.clip(upper=0)).ewm(alpha=1 / n, adjust=False).mean()
    return 100 - 100 / (1 + up / dn.replace(0, np.nan))

def _regime_window_mean(x, vix, L_low, L_high, thr):
    lo = x.rolling(L_low, min_periods=L_low).mean()
    hi = x.rolling(L_high, min_periods=L_high).mean()
    return pd.Series(np.where(vix >= thr, hi, lo), index=x.index)

def _regime_window_sum(x, vix, L_low, L_high, thr):
    lo = x.rolling(L_low, min_periods=1).sum()
    hi = x.rolling(L_high, min_periods=1).sum()
    return pd.Series(np.where(vix >= thr, hi, lo), index=x.index)

def build_market_features(market):
    m = pd.DataFrame(index=market.index)
    lr = market['mkt_logret']
    Lm = LOOKBACKS['sp500_return']['L']
    m['sp500_ret_1d'] = lr
    m[f'sp500_ret_{Lm}d'] = lr.rolling(Lm).sum()
    for h in [21, 63]:
        m[f'sp500_ret_{h}d'] = market['sp500'] / market['sp500'].shift(h) - 1
    m['sp500_trend_200'] = market['sp500'] / market['sp500'].rolling(200).mean() - 1
    m['sp500_vol_21d'] = lr.rolling(21).std()
    # VIX
    Lv = LOOKBACKS['log_vix']['L']
    if INTEGRATION_ORDER['log_vix'] == 0:
        m['log_vix'] = market['log_vix']
    else:
        m[f'log_vix_chg{Lv}d'] = market['log_vix'].diff(Lv)
    m[f'vix_z_{Lv}d'] = ((market['log_vix'] - market['log_vix'].rolling(Lv).mean())
                         / market['log_vix'].rolling(Lv).std())
    m[f'vix_chg_{HORIZON}d'] = market['log_vix'].diff(HORIZON)
    m['vix_term'] = market['vix'] / market['vix3m'] - 1            # > 0 = inverted (stress)
    m['vrp'] = (market['vix'] / 100) ** 2 / 12 - (lr ** 2).rolling(21).sum()   # Bollerslev-Tauchen-Zhou; both terms are ONE-MONTH variances:
    # VIX^2/12 = annualised implied variance / 12, and the sum of 21 squared daily log returns (21 trading days = 1/12 year)
    m['vix_level'] = market['vix']
    # expanding percentile of VIX using past values only (used for position scaling, Section 7a)
    v = market['vix'].values
    pct = np.full(len(v), np.nan)
    for i in range(252, len(v)):
        past = v[:i]
        past = past[~np.isnan(past)]
        if len(past) and not np.isnan(v[i]):
            pct[i] = (past <= v[i]).mean()
    m['vix_pctile'] = pct
    # macro: level if I(0), L-day change if I(1)
    for var in ['ust10y', 'term_spread', 'credit_spread', 'breakeven10y', 'log_usd_broad', 'log_brent']:
        L = LOOKBACKS[var]['L']
        if INTEGRATION_ORDER[var] == 0:
            m[var] = market[var]
        else:
            m[f'{var}_chg{L}d'] = market[var].diff(L)
    m['_sp500_close'] = market['sp500']
    return m

def build_stock_features(panel, mkt_feat):
    lr_lb = LOOKBACKS['stock_return']
    out = []
    for t, g in panel.groupby('ticker', sort=False):
        g = g.sort_values('date').set_index('date').copy()
        c, lr = g['Close'], g['logret']
        mret = mkt_feat['sp500_ret_1d'].reindex(g.index)
        vix = market['vix'].reindex(g.index).ffill().values
        g['ret_1d'] = c.pct_change()
        for h in [5, 10, 21, 63]:
            g[f'ret_{h}d'] = c / c.shift(h) - 1
        g['mom_12_1'] = c.shift(21) / c.shift(252) - 1
        g['vol_21d'] = lr.rolling(21).std()
        g['vol_63d'] = lr.rolling(63).std()
        g['vol_ratio_21_63'] = g['vol_21d'] / g['vol_63d']
        g['vol_sizing'] = g['vol_21d']
        g['rsi_14'] = _rsi(c)
        ema12, ema26 = c.ewm(span=12, adjust=False).mean(), c.ewm(span=26, adjust=False).mean()
        macd = ema12 - ema26
        g['macd_norm'] = macd / c
        g['macd_hist_norm'] = (macd - macd.ewm(span=9, adjust=False).mean()) / c
        for w in [10, 20, 50, 200]:
            g[f'sma_{w}_ratio'] = c / c.rolling(w).mean() - 1
        g['ma_cross_50_200'] = c.rolling(50).mean() / c.rolling(200).mean() - 1
        g['dist_52w_high'] = c / g['High'].rolling(252).max() - 1
        g['dist_52w_low'] = c / g['Low'].rolling(252).min() - 1
        g['log_volume_ratio'] = np.log(g['Volume'].replace(0, np.nan) / g['Volume'].rolling(21).mean())
        dv = (c * g['Volume']).replace(0, np.nan)
        g['amihud_21d'] = np.log1p((lr.abs() / dv).rolling(21).mean() * 1e9)
        beta = lr.rolling(252).cov(mret) / mret.rolling(252).var()
        g['beta_252d'] = beta
        g['idio_vol_63d'] = (lr - beta * mret).rolling(63).std()
        spx = mkt_feat['_sp500_close'].reindex(g.index)
        g['rel_strength_21d'] = g['ret_21d'] - (spx / spx.shift(21) - 1)
        for k in range(1, P_RET + 1):
            g[f'ret_lag{k}'] = lr.shift(k - 1)
        g['ret_mean_regime'] = _regime_window_mean(lr, vix, lr_lb['L_low'], lr_lb['L_high'], lr_lb['vix_threshold'])
        out.append(g.reset_index())
    return pd.concat(out, ignore_index=True)

mkt_feat = build_market_features(market)
stock_feat = build_stock_features(panel, mkt_feat)
print(f"stock features: {stock_feat.shape} | market features: {mkt_feat.shape}")

In [ ]:
def arima_walkforward_features(y, order, min_train=500):
    """One-step-ahead ARIMA forecast of tomorrow's return and today's innovation.
    Parameters for calendar year Y are estimated on data up to Dec 31 of Y-1 and then held
    fixed while the Kalman filter runs through year Y (res.apply(..., refit=False)).
    fitted[t] = E[y_t | y_<t]; so fitted[t+1] = E[y_{t+1} | y_<=t] is known at the close of t."""
    fc = pd.Series(np.nan, index=y.index)
    innov = pd.Series(np.nan, index=y.index)
    for yr in sorted(set(y.index.year)):
        train = y[y.index < pd.Timestamp(f'{yr}-01-01')]
        if len(train) < min_train:
            continue
        res, _ = _fit_arima(train.values, order)
        upto = y[y.index <= pd.Timestamp(f'{yr}-12-31')]
        filt = res.apply(upto.values, refit=False)
        fitted = pd.Series(np.asarray(filt.fittedvalues), index=upto.index)
        nxt = fitted.shift(-1)
        nxt.iloc[-1] = float(np.asarray(filt.forecast(1))[0])
        in_year = upto.index.year == yr
        fc[upto.index[in_year]] = nxt[in_year].values
        innov[upto.index[in_year]] = (upto - fitted)[in_year].values
    return fc, innov

t0 = time.time()
arima_parts = []
for t, g in panel.groupby('ticker'):
    y = (g.set_index('date')['logret'] * 100).dropna()
    fc, innov = arima_walkforward_features(y, ARIMA_ORDERS[t])
    arima_parts.append(pd.DataFrame({'date': y.index, 'ticker': t,
                                     'arima_fcst_1d': fc.values, 'arima_innov': innov.values}))
arima_feat = pd.concat(arima_parts, ignore_index=True)
print(f"ARIMA features built in {time.time() - t0:.1f}s; coverage: "
      f"{arima_feat['arima_fcst_1d'].notna().mean():.1%} of rows (first year per stock is warm-up)")

# Leakage unit test: changing a FUTURE return must not change any earlier forecast.
_t = TICKERS[0]
_y = (panel[panel.ticker == _t].set_index('date')['logret'] * 100).dropna()
_y = _y[_y.index.year <= _y.index.year.min() + 3]
_fc_a, _ = arima_walkforward_features(_y, ARIMA_ORDERS[_t])
_cut = _y.index[-40]
_y2 = _y.copy(); _y2[_y2.index > _cut] += 5.0
_fc_b, _ = arima_walkforward_features(_y2, ARIMA_ORDERS[_t])
assert np.allclose(_fc_a[_fc_a.index <= _cut].dropna(), _fc_b[_fc_b.index <= _cut].dropna()), "ARIMA feature leaks"
print("Leakage test passed: ARIMA forecasts dated <= t are unchanged when returns after t are altered.")

In [ ]:
def attach_fundamentals(df, fund_q):
    cols = ['eps_ttm', 'eps_growth_yoy', 'rev_growth_yoy', 'roe', 'net_margin', 'sue']
    if fund_q is None:
        for c in cols + ['earnings_yield', 'days_since_filing']:
            df[c] = np.nan
        return df
    f = fund_q[['ticker', 'available_date'] + cols].dropna(subset=['available_date']).sort_values('available_date')
    df = df.sort_values('date')
    df = pd.merge_asof(df, f, left_on='date', right_on='available_date', by='ticker', direction='backward')
    df['days_since_filing'] = (df['date'] - df['available_date']).dt.days
    stale = df['days_since_filing'] > 200            # no filing for > 2 quarters: treat as missing
    df.loc[stale, cols] = np.nan
    df['earnings_yield'] = (df['eps_ttm'] / df['Close']).clip(-0.5, 0.5)
    assert (df['available_date'].isna() | (df['available_date'] <= df['date'])).all()
    return df.drop(columns=['available_date'])

def attach_sentiment(df, sent_daily):
    df = df.merge(sent_daily, on=['ticker', 'date'], how='left')
    df[['finbert_sum', 'qwen_sum', 'n_headlines']] = df[['finbert_sum', 'qwen_sum', 'n_headlines']].fillna(0.0)
    out = []
    for t, g in df.groupby('ticker', sort=False):
        g = g.sort_values('date').copy()
        vix = market['vix'].reindex(g['date']).ffill().values
        for src in ['finbert', 'qwen']:
            lb = LOOKBACKS[f'{src}_daily']
            s = _regime_window_sum(g[f'{src}_sum'], vix, lb['L_low'], lb['L_high'], lb['vix_threshold'])
            n = _regime_window_sum(g['n_headlines'], vix, lb['L_low'], lb['L_high'], lb['vix_threshold'])
            g[f'avg_{src}_sentiment'] = np.where(n > 0, s / n.clip(lower=1), 0.0)
            g[f'{src}_has_news'] = (n > 0).astype(float)
        g['has_news_window'] = (g['n_headlines'].rolling(HORIZON, min_periods=1).sum() > 0).astype(float)
        out.append(g)
    df = pd.concat(out, ignore_index=True)
    df['sentiment_available'] = (df['date'] >= pd.Timestamp(SENTIMENT_START)).astype(float)
    return df

def add_labels(df, horizon):
    out = []
    for t, g in df.groupby('ticker', sort=False):
        g = g.sort_values('date').copy()
        g['fwd_return'] = g['Close'].shift(-horizon) / g['Close'] - 1
        out.append(g)
    df = pd.concat(out, ignore_index=True)
    spx = mkt_feat['_sp500_close']
    df['sp500_fwd_return'] = (spx.shift(-horizon) / spx - 1).reindex(df['date']).values
    if LABEL_MODE == 'cross_sectional':
        med = df.groupby('date')['fwd_return'].transform('median')
        df['target'] = np.where(df['fwd_return'].isna(), np.nan, (df['fwd_return'] > med).astype(float))
    else:
        df['target'] = np.where(df['fwd_return'].isna(), np.nan, (df['fwd_return'] > 0).astype(float))
    return df

def xs_rank(df, col):
    r = df.groupby('date')[col].rank(pct=True)
    n = df.groupby('date')[col].transform('count')
    return (r - 0.5).where(n >= 2).fillna(0.0)

df = stock_feat.merge(mkt_feat.drop(columns=['_sp500_close']).reset_index(), on='date', how='left')
df = df.merge(arima_feat, on=['date', 'ticker'], how='left')
df = attach_fundamentals(df, fund_q)
df = attach_sentiment(df, sent_daily)
df = add_labels(df, HORIZON)

FUND_RAW = ['earnings_yield', 'eps_growth_yoy', 'rev_growth_yoy', 'roe', 'net_margin', 'sue']
df['fund_available'] = df['eps_ttm'].notna().astype(float)
for c in FUND_RAW:
    df[f'{c}_xsr'] = xs_rank(df, c)
df['log_days_since_filing'] = np.log1p(df['days_since_filing'].clip(lower=0)).fillna(0.0)
for c in ['ret_5d', 'ret_21d', 'mom_12_1']:
    df[f'{c}_xs'] = df[c] - df.groupby('date')[c].transform('mean')

df = df[df['date'] >= pd.Timestamp(MODEL_START)].sort_values(['ticker', 'date']).reset_index(drop=True)
print(f"{len(df):,} rows | {df['ticker'].nunique()} tickers | {df['date'].min().date()} to {df['date'].max().date()}")

## 5. Feature sets

`BASE_FEATURES` is the union of all non-sentiment families (56 features). Three configurations are compared: **baseline**, **baseline + FinBERT** and **baseline + Qwen**. The sentiment extras are the source's average score over its own data-driven look-back, its has-news flag and a sentiment-availability flag (so the model can tell "no news" apart from "before the headline dataset begins").

### 5.1 Handling 56 features

All 56 base features go to every model, and selection happens inside the model through regularisation, following Gu, Kelly and Xiu (2020):

* Ridge logistic regression: L2 penalty strength `C` tuned by purged cross-validation.
* XGBoost: column subsampling, `min_child_weight`, L2 on leaf weights and early stopping. The tree-count and sentiment-gain diagnostics in Section 11 show what the model actually used.
* LSTM: dropout, weight decay and early stopping.

The information-coefficient table in Section 6 is a diagnostic on pre-out-of-sample data only. No feature is dropped because of it, since that would be selection on evaluation data.

**Caveat.** Gu, Kelly and Xiu have thousands of stocks per month, while this panel has 7 highly correlated stocks and overlapping 5-day labels. The effective number of independent observations is therefore far smaller than the roughly 23,600 rows suggest, and the overfitting risk per feature is higher. The train-test gap table (12e) is the first check on this.

In [ ]:
_lr = LOOKBACKS
FEATURE_FAMILIES = {
    'momentum_reversal': ['ret_1d', 'ret_5d', 'ret_10d', 'ret_21d', 'ret_63d', 'mom_12_1',
                          'ret_5d_xs', 'ret_21d_xs', 'mom_12_1_xs'],
    'autoregressive': [f'ret_lag{k}' for k in range(2, P_RET + 1)] + ['ret_mean_regime', 'arima_fcst_1d', 'arima_innov'],
    'trend': ['sma_10_ratio', 'sma_20_ratio', 'sma_50_ratio', 'sma_200_ratio', 'ma_cross_50_200',
              'dist_52w_high', 'dist_52w_low', 'macd_norm', 'macd_hist_norm', 'rsi_14'],
    'risk_liquidity': ['vol_21d', 'vol_63d', 'vol_ratio_21_63', 'beta_252d', 'idio_vol_63d',
                       'log_volume_ratio', 'amihud_21d'],
    'market_sp500': [c for c in mkt_feat.columns if c.startswith('sp500_')] + ['rel_strength_21d'],
    'vix': [c for c in mkt_feat.columns if c.startswith(('log_vix', 'vix_z', 'vix_chg', 'vix_term'))] + ['vrp', 'vix_pctile'],
    'macro': [c for c in mkt_feat.columns if c.startswith(('ust10y', 'term_spread', 'credit_spread',
                                                          'breakeven10y', 'log_usd_broad', 'log_brent'))],
    'fundamental': [f'{c}_xsr' for c in FUND_RAW] + ['log_days_since_filing', 'fund_available'],
}
if not USE_FUNDAMENTALS:
    FEATURE_FAMILIES.pop('fundamental')
BASE_FEATURES = [f for fam in FEATURE_FAMILIES.values() for f in fam]
assert len(BASE_FEATURES) == len(set(BASE_FEATURES)), "duplicate feature names"
FINBERT_EXTRA = ['avg_finbert_sentiment', 'finbert_has_news', 'sentiment_available']
QWEN_EXTRA = ['avg_qwen_sentiment', 'qwen_has_news', 'sentiment_available']
FEATURE_SETS = {'baseline': BASE_FEATURES,
                'finbert': BASE_FEATURES + FINBERT_EXTRA,
                'qwen': BASE_FEATURES + QWEN_EXTRA}

missing = [f for f in BASE_FEATURES + FINBERT_EXTRA + QWEN_EXTRA if f not in df.columns]
assert not missing, f"features not built: {missing}"
for fam, cols in FEATURE_FAMILIES.items():
    print(f"{fam:<18} ({len(cols):>2}): {', '.join(cols)}")
print(f"\n{len(BASE_FEATURES)} base features (v12: 18), +{len(FINBERT_EXTRA)} per sentiment source")

usable = df.dropna(subset=BASE_FEATURES + ['target'])
print(f"Usable model rows: {len(usable):,} | first usable date per ticker:")
print(usable.groupby('ticker')['date'].min().dt.date.to_string())
na_share = df[df['date'] >= usable['date'].min()][BASE_FEATURES].isna().mean().sort_values(ascending=False)
print("\nHighest NaN shares after the global warm-up (rows with NaN are dropped per ticker):")
print(na_share.head(8).round(3).to_string())

## 6. Exploratory analysis and signal diagnostics (before any model fitting)

### 6a. Temporal dependence (bootstrap check)

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

N_BOOT = 1000
BLOCK_SIZE = 10
BOOT_SEED = RANDOM_SEED

def iid_bootstrap_se(x, n_boot=N_BOOT, seed=BOOT_SEED):
    rng = np.random.default_rng(seed)
    return np.std([rng.choice(x, size=len(x), replace=True).mean() for _ in range(n_boot)])

def block_bootstrap_se(x, block_size=BLOCK_SIZE, n_boot=N_BOOT, seed=BOOT_SEED):
    rng = np.random.default_rng(seed)
    n = len(x); max_start = n - block_size
    if max_start < 1:
        return np.nan
    nb = int(np.ceil(n / block_size)); means = []
    for _ in range(n_boot):
        starts = rng.integers(0, max_start + 1, size=nb)
        means.append(np.concatenate([x[s:s + block_size] for s in starts])[:n].mean())
    return np.std(means)

rows = []
print(f"{'Ticker':<8}{'IID SE':>12}{'Block SE':>12}{'Ratio':>10}{'Ljung-Box p (lag 5)':>22}")
for t in TICKERS:
    x = df.loc[df.ticker == t, 'ret_1d'].dropna().values
    if len(x) < 50:
        continue
    iid_se, block_se = iid_bootstrap_se(x), block_bootstrap_se(x)
    lb_p = acorr_ljungbox(x, lags=[5], return_df=True)['lb_pvalue'].iloc[0]
    rows.append(dict(ticker=t, iid_se=iid_se, block_se=block_se, ratio=block_se / iid_se, ljung_box_p=lb_p))
    print(f"{t:<8}{iid_se:>12.5f}{block_se:>12.5f}{block_se / iid_se:>10.3f}{lb_p:>22.4f}")
temporal_dependence_df = pd.DataFrame(rows)
temporal_dependence_df.to_csv('temporal_dependence_bootstrap_check.csv', index=False)

### 6b. FinBERT vs. Qwen (sentiment era only, de-overlapped)

Tests of agreement between the two sources, restricted to the sentiment era (from `SENTIMENT_START`) and to the rebalance calendar, so the paired tests use non-overlapping windows.

In [ ]:
from scipy.stats import ttest_rel, wilcoxon, pearsonr, spearmanr

era = df[df['date'] >= pd.Timestamp(SENTIMENT_START)]
news_days = era[(era['finbert_has_news'] == 1) & (era['qwen_has_news'] == 1)].copy()
print(f"Sentiment-era rows: {len(era):,}; rows with news in both sources' windows: {len(news_days):,} "
      f"({len(news_days) / len(era):.1%})")
print(news_days[['avg_finbert_sentiment', 'avg_qwen_sentiment']].describe().round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(news_days['avg_finbert_sentiment'], bins=40, alpha=0.6, label='FinBERT')
axes[0].hist(news_days['avg_qwen_sentiment'], bins=40, alpha=0.6, label='Qwen')
axes[0].set_title('Sentiment score distributions (news days)'); axes[0].legend()
axes[1].scatter(news_days['avg_finbert_sentiment'], news_days['avg_qwen_sentiment'], alpha=0.15, s=8)
axes[1].set_xlabel('FinBERT'); axes[1].set_ylabel('Qwen'); axes[1].set_title('FinBERT vs Qwen, paired')
plt.tight_layout(); plt.savefig('eda_sentiment_distributions.png', dpi=120, bbox_inches='tight'); plt.show()

rebalance_calendar = set(np.sort(df['date'].unique())[::HORIZON])
nd = news_days[news_days['date'].isin(rebalance_calendar)]
fb, qw = nd['avg_finbert_sentiment'].values, nd['avg_qwen_sentiment'].values
t_stat, t_p = ttest_rel(fb, qw); w_stat, w_p = wilcoxon(fb, qw)
pr, pp = pearsonr(fb, qw); sr, sp = spearmanr(fb, qw)
print(f"\nDe-overlapped sample: {len(nd):,} rows")
print(f"Paired t-test        : t={t_stat:.3f}, p={t_p:.4g}")
print(f"Wilcoxon signed-rank : W={w_stat:.1f}, p={w_p:.4g}")
print(f"Pearson / Spearman   : r={pr:.3f} (p={pp:.3g}) / rho={sr:.3f} (p={sp:.3g})")
pd.DataFrame([dict(test='paired_t', statistic=t_stat, p_value=t_p), dict(test='wilcoxon', statistic=w_stat, p_value=w_p),
              dict(test='pearson_r', statistic=pr, p_value=pp), dict(test='spearman_rho', statistic=sr, p_value=sp)]
             ).to_csv('finbert_vs_qwen_statistical_comparison.csv', index=False)

### 6c. Information coefficient by feature

For each feature: the mean over rebalance dates of the cross-sectional Spearman correlation with the 5-day forward return (**rank IC**), with a Newey-West t-statistic, plus the pooled time-series Pearson IC. Computed on the **pre-OOS window only** (sentiment: its first year of data). This is a diagnostic only. No feature is selected or dropped using it, so it cannot leak into the out-of-sample results.

The second table shows the horizon on which the signal lives, as evidence for the 5-day horizon: median |rank IC| across all base features for 1, 5, 10 and 21-day forward returns.

In [ ]:
import statsmodels.api as sm

def daily_rank_ic(frame, score_col, ret_col):
    d = frame[['date', score_col, ret_col]].dropna()
    if d.empty:
        return pd.Series(dtype=float)
    rs = d.groupby('date')[score_col].rank(); rf = d.groupby('date')[ret_col].rank()
    rs_c = rs - rs.groupby(d['date']).transform('mean'); rf_c = rf - rf.groupby(d['date']).transform('mean')
    num = (rs_c * rf_c).groupby(d['date']).sum()
    den = np.sqrt((rs_c ** 2).groupby(d['date']).sum() * (rf_c ** 2).groupby(d['date']).sum())
    cnt = d.groupby('date').size()
    return (num / den.where(den > 0)).where(cnt >= 3).dropna()

def hac_tstat(x, lags=4):
    x = pd.Series(x).dropna()
    if len(x) < 10 or x.std() == 0:
        return np.nan
    fit = sm.OLS(x.values, np.ones(len(x))).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    return float(fit.tvalues[0])

ic_frame = df[df['date'].isin(rebalance_calendar)]
ic_rows = []
for f in BASE_FEATURES + ['avg_finbert_sentiment', 'avg_qwen_sentiment']:
    is_sent = 'sentiment' in f
    sub = ic_frame[(ic_frame['date'] >= pd.Timestamp(SENTIMENT_START)) & (ic_frame['date'] < pd.Timestamp(SENTIMENT_SELECTION_END))] \
        if is_sent else ic_frame[ic_frame['date'] < FIRST_OOS_DATE]
    ics = daily_rank_ic(sub, f, 'fwd_return')
    pooled = sub[[f, 'fwd_return']].dropna()
    pr_ = pearsonr(pooled[f], pooled['fwd_return'])[0] if len(pooled) > 10 and pooled[f].std() > 0 else np.nan
    fam = next((k for k, v in FEATURE_FAMILIES.items() if f in v), 'sentiment')
    ic_rows.append(dict(feature=f, family=fam, mean_rank_ic=ics.mean(), rank_ic_t_hac=hac_tstat(ics),
                        pooled_pearson_ic=pr_, n_dates=len(ics)))
ic_df = pd.DataFrame(ic_rows).sort_values('mean_rank_ic', key=abs, ascending=False).reset_index(drop=True)
ic_df.to_csv('information_coefficient.csv', index=False)
print(ic_df.round(4).to_string(index=False))
print(f"\n{(ic_df['rank_ic_t_hac'].abs() > 2).sum()}/{len(ic_df)} features with |HAC t| > 2 on the selection window")
_small = ic_df[(ic_df['n_dates'] > 0) & (ic_df['n_dates'] < 60)]   # n_dates == 0: cross-sectionally constant feature, IC undefined
if len(_small):
    print(f"CAUTION: {len(_small)} feature(s) ({', '.join(_small['feature'])}) have fewer than 60 IC dates. The HAC t-statistic "
          f"uses a fixed 4 lags regardless of sample size, so with this few dates it is unreliable; read those rows as descriptive only.")

fig, ax = plt.subplots(figsize=(9, 12))
fam_colors = dict(zip(list(FEATURE_FAMILIES) + ['sentiment'], sns.color_palette('tab10', len(FEATURE_FAMILIES) + 1)))
ax.barh(ic_df['feature'][::-1], ic_df['mean_rank_ic'][::-1], color=[fam_colors[f] for f in ic_df['family'][::-1]])
ax.axvline(0, color='black', lw=0.8); ax.set_xlabel('mean cross-sectional rank IC (pre-OOS)')
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in fam_colors.values()]
ax.legend(handles, fam_colors.keys(), fontsize=7, loc='lower right')
ax.set_title('Rank IC by feature, coloured by family'); plt.tight_layout()
plt.savefig('eda_information_coefficient.png', dpi=120, bbox_inches='tight'); plt.show()

# horizon check
hz_rows = []
tmp = df[df['date'] < FIRST_OOS_DATE].copy()
for h in [1, 5, 10, 21]:
    tmp[f'fwd_{h}'] = tmp.groupby('ticker')['Close'].transform(lambda c: c.shift(-h) / c - 1)
    cal = set(np.sort(tmp['date'].unique())[::h])
    sub = tmp[tmp['date'].isin(cal)]
    vals = [daily_rank_ic(sub, f, f'fwd_{h}').mean() for f in BASE_FEATURES]
    hz_rows.append(dict(horizon_days=h, median_abs_rank_ic=np.nanmedian(np.abs(vals)),
                        n_features_abs_ic_gt_0_03=int(np.sum(np.abs(vals) > 0.03))))
print("\nSignal strength by forecast horizon (pre-OOS):")
print(pd.DataFrame(hz_rows).round(4).to_string(index=False))

### 6d. Target class balance

In [ ]:
target_valid = df.dropna(subset=['target'])
print(f"Label mode: {LABEL_MODE}. Overall share of class 1: {target_valid['target'].mean():.1%}")
per_ticker_balance = target_valid.groupby('ticker')['target'].mean().sort_values(ascending=False)
print((per_ticker_balance * 100).round(1).astype(str) + '%')
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(per_ticker_balance.index, per_ticker_balance.values * 100)
ax.axhline(50, color='grey', ls='--', lw=1); ax.set_ylabel('% of rows labelled 1')
ax.set_title(f'Target class balance by ticker ({LABEL_MODE} label)'); plt.tight_layout()
plt.savefig('eda_class_balance.png', dpi=120, bbox_inches='tight'); plt.show()

### 6e. Buy-and-hold over the full 16-year sample

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
wide = eq_raw.pivot(index='date', columns='symbol', values='Close')
daily_ret_wide = wide.pct_change()
for t in TICKERS:
    cum = (1 + daily_ret_wide[t].fillna(0)).cumprod()
    ax.plot(cum.index, cum.values, label=t, lw=1.0)
pooled_cum = (1 + daily_ret_wide.mean(axis=1).fillna(0)).cumprod()
ax.plot(pooled_cum.index, pooled_cum.values, label='Equal-weight Mag 7', color='black', lw=2.2, ls='--')
spx = (1 + market['sp500'].pct_change().fillna(0)).cumprod()
ax.plot(spx.index, spx.values, label='S&P 500', color='grey', lw=2)
ax.set_yscale('log')
_marks = [(FIRST_OOS_DATE, 'first OOS date')]
if pd.Timestamp(SENTIMENT_EVAL_START) > pd.Timestamp(FIRST_OOS_DATE):
    _marks.append((pd.Timestamp(SENTIMENT_EVAL_START), 'sentiment comparisons start'))
for dt, lab in _marks:
    ax.axvline(dt, color='grey', ls=':', lw=1.5)
    ax.annotate(lab, xy=(dt, ax.get_ylim()[1]), xytext=(4, -12), textcoords='offset points', fontsize=8, color='grey')
ax.set_ylabel('growth of $1 (log scale)'); ax.set_title('Buy-and-hold, 2010-2025')
ax.legend(ncol=5, fontsize=8, loc='upper left'); plt.tight_layout()
plt.savefig('eda_cumulative_return_full_period.png', dpi=120, bbox_inches='tight'); plt.show()

### 6f. Return vs. sentiment, per stock (sentiment era)

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(13, 14)); axes = axes.flatten()
for i, t in enumerate(TICKERS):
    g = df[(df.ticker == t) & (df.date >= pd.Timestamp(SENTIMENT_START))].sort_values('date')
    cum = (1 + g['ret_1d'].fillna(0)).cumprod() - 1
    ax1 = axes[i]; ax1.plot(g['date'], cum * 100, color='tab:blue', lw=1)
    ax1.set_ylabel('Cum. return (%)', color='tab:blue', fontsize=8); ax1.tick_params(labelsize=7)
    ax2 = ax1.twinx()
    ax2.plot(g['date'], g['avg_finbert_sentiment'].rolling(20, min_periods=1).mean(), color='tab:orange', lw=0.8)
    ax2.plot(g['date'], g['avg_qwen_sentiment'].rolling(20, min_periods=1).mean(), color='tab:green', lw=0.8)
    ax2.tick_params(labelsize=7); ax1.set_title(t, fontsize=10)
axes[-1].axis('off'); plt.tight_layout()
plt.savefig('eda_return_vs_sentiment_per_stock.png', dpi=120, bbox_inches='tight'); plt.show()

### 6g. Fundamentals over time (EPS TTM and earnings yield)

In [ ]:
if USE_FUNDAMENTALS:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    for t in TICKERS:
        g = df[df.ticker == t].sort_values('date')
        axes[0].plot(g['date'], g['eps_ttm'], lw=1, label=t)
        axes[1].plot(g['date'], g['earnings_yield'] * 100, lw=1, label=t)
    axes[0].set_title('Diluted EPS, trailing 12 months (split-adjusted, as first reported)')
    axes[1].set_title('Earnings yield (%) = EPS TTM / price'); axes[1].axhline(0, color='grey', lw=0.8)
    axes[0].legend(ncol=4, fontsize=7); plt.tight_layout()
    plt.savefig('eda_fundamentals.png', dpi=120, bbox_inches='tight'); plt.show()

## 7. Shared evaluation engine

### 7a. Portfolio construction

Applied identically to all three architectures (Jansen, 2020):

1. **Selection.** On every rebalance date, hold the `PORTFOLIO_TOP_K` = 3 names with the highest model score (`PORTFOLIO_MODE = 'rank'`). Because the label is cross-sectional (beat the day's median), the model's job is to rank the seven names, and the book is almost always invested, so differences come from stock selection and not from time in cash. A `'threshold'` mode also exists in the code but is not used for any result.
2. **Inverse-volatility weights**, $w_i \propto 1/\sigma_{i,21d}$, normalised to a gross exposure of 1 and capped at `PORTFOLIO_MAX_WEIGHT` (excess stays in cash).
3. **VIX regime scaling.** Exposure is scaled down as the VIX moves from its own expanding 70th to 95th percentile (past data only), to a floor of 30%. This follows the volatility-management literature (Moreira & Muir, 2017, Barroso & Santa-Clara, 2015). The percentile implementation and its 70/95/30% parameters are my own and were fixed in advance (see 3.0).
4. **Drawdown circuit breaker** on the pooled book: flat after a 15% drawdown of the signal's equity curve, back in once that drawdown is below 7%. Re-entry is measured on the unscaled curve, so a flat book can recover.

Portfolio return per rebalance date = sum over names of weight x 5-day forward return, net of 5 bps per unit of turnover.

### 7b. Leakage controls (purging as in `MultipleTimeSeriesCV`)

* **Label buffer.** Training rows end `PURGE_DAYS` = 5 trading days before each test block, so no training label overlaps the test period. The same buffer is applied inside hyperparameter cross-validation and the early-stopping split.
* **Hyperparameters.** Tuned once on the initial window with purged, date-based folds and, with `SHARED_HYPERPARAMS = True`, once per model family on the baseline feature set. The same settings are reused for +FinBERT and +Qwen. The initial window contains at most one year of sentiment, so a separate search for the sentiment sets could not learn anything about sentiment and would only add tuning noise.
* **Thresholds.** Used only in the unused `'threshold'` mode. They are re-selected at every refit on the preceding year of realised out-of-sample predictions.

In [ ]:
"""Shared walk-forward engine used by all three model families."""
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, precision_recall_fscore_support

META_COLS = ['date', 'ticker', 'fwd_return', 'sp500_fwd_return', 'target', 'has_news_window', 'vol_sizing', 'vix_pctile']
TRAIN_WINDOW_DAYS = None if TRAIN_WINDOW_YEARS is None else int(252 * TRAIN_WINDOW_YEARS)
PERIODS_PER_YEAR = 252 / HORIZON

def prepare_tabular(df, feature_cols):
    d = df[df.ticker.isin(TICKERS)].dropna(subset=feature_cols + ['target', 'fwd_return', 'vol_sizing'])
    return d.sort_values(['date', 'ticker']).reset_index(drop=True)

def add_ticker_dummies(d):
    dummies = pd.get_dummies(d['ticker'], prefix='tk', drop_first=True).astype(float)
    return pd.concat([d.reset_index(drop=True), dummies.reset_index(drop=True)], axis=1), list(dummies.columns)

def n_initial_days(dates):
    return int(np.searchsorted(np.sort(dates), np.datetime64(FIRST_OOS_DATE)))

# ---------------------------------------------------------------------------
# Walk-forward with label buffer
# ---------------------------------------------------------------------------
def walk_forward(d, feature_cols, fit_predict_fn, initial_train_days, retrain_step_days,
                 purge=PURGE_DAYS, train_window_days=TRAIN_WINDOW_DAYS):
    dates = np.sort(d['date'].unique())
    oos_proba, oos_meta = [], []
    start = initial_train_days
    while start < len(dates):
        end = min(start + retrain_step_days, len(dates))
        train_end = start - purge
        train_start = 0 if train_window_days is None else max(0, train_end - train_window_days)
        train_df = d[(d.date >= dates[train_start]) & (d.date < dates[train_end])]
        test_df = d[(d.date >= dates[start]) & (d.date <= dates[end - 1])]
        assert train_df.empty or train_df['date'].max() < dates[train_end], 'label buffer violated'
        if len(test_df) and train_df['target'].nunique() == 2:
            oos_proba.append(fit_predict_fn(train_df, test_df, feature_cols))
            m = test_df[META_COLS].copy(); m['block_start'] = dates[start]
            oos_meta.append(m)
        start = end
    return np.concatenate(oos_proba), pd.concat(oos_meta, ignore_index=True)

def get_validation_split(d, initial_train_days, val_frac=0.2, purge=PURGE_DAYS):
    """Train/validation split inside the INITIAL window only, with a label buffer between them
    and validation labels fully realised before the first OOS date."""
    dates = np.sort(d['date'].unique())
    split = int(initial_train_days * (1 - val_frac))
    train_df = d[d.date < dates[split - purge]]
    val_df = d[(d.date >= dates[split]) & (d.date < dates[initial_train_days - purge])]
    return train_df, val_df

def purged_date_folds(dates, n_splits=3, purge=PURGE_DAYS, min_train_frac=0.4):
    """Expanding, date-based walk-forward folds with a label buffer, for hyperparameter tuning.
    Yields boolean masks aligned with `dates` (rows), so pooled multi-ticker rows on the same
    date always fall on the same side of a split."""
    u = np.sort(np.unique(dates)); n = len(u)
    first = int(n * min_train_frac); fold = (n - first) // n_splits
    for k in range(n_splits):
        vs = first + k * fold
        ve = n if k == n_splits - 1 else vs + fold
        yield np.isin(dates, u[:vs - purge]), np.isin(dates, u[vs:ve])

# ---------------------------------------------------------------------------
# Signal metrics
# ---------------------------------------------------------------------------
def mean_rank_ic(score, dates, fwd):
    frame = pd.DataFrame({'date': dates, 's': score, 'f': fwd})
    return daily_rank_ic(frame, 's', 'f').mean()

def tuning_score(y, proba, dates, fwd):
    if TUNING_METRIC == 'rank_ic':
        v = mean_rank_ic(proba, dates, fwd)
        return 0.0 if pd.isna(v) else v
    return f1_score(y, (proba >= 0.5).astype(int), average='macro')

# ---------------------------------------------------------------------------
# Portfolio construction
# ---------------------------------------------------------------------------
def _tie_key(bt):
    # Deterministic pseudo-random key in [0, 1) from (date, ticker), independent of row order. Exactly tied model scores (common when
    # early stopping keeps 1-2 trees) are broken with it, instead of by row order, which is ALPHABETICAL by ticker and would
    # systematically favour AAPL/AMZN/GOOGL over NVDA/TSLA.
    h = pd.util.hash_pandas_object(bt['date'].astype(str) + '|' + bt['ticker'].astype(str), index=False).values
    return (h % np.uint64(1000003)).astype(float) / 1000003.0

def build_position_weights(bt, top_k=PORTFOLIO_TOP_K, max_weight=PORTFOLIO_MAX_WEIGHT,
                           soft=VIX_PCTILE_SOFT, hard=VIX_PCTILE_HARD, min_scale=VIX_MIN_SCALE):
    """bt: one row per (ticker, rebalance date) with proba, thr_long, thr_short, edge, vol_sizing,
    vix_pctile. Adds signed 'weight' (portfolio weight, gross <= 1) and 'position'."""
    bt = bt.copy()
    bt['_score'] = bt['proba'] + 1e-9 * _tie_key(bt)      # ranking score = proba with a negligible random tie-break
    if PORTFOLIO_MODE == 'rank':
        # Ranking strategy: every stock is a long candidate (top-K chosen below).
        # In long-short mode the bottom-K stocks are the short candidates.
        sig = np.ones(len(bt), dtype=int)
        if not LONG_ONLY_MODE:
            r_asc = bt.groupby('date')['_score'].rank(method='first').values
            n_d = bt.groupby('date')['proba'].transform('count').values
            sig = np.where(r_asc <= top_k, -1, np.where(r_asc > n_d - top_k, 1, 0))
    else:
        sig = np.where(bt['proba'].values > bt['thr_long'].values, 1, 0)
        if not LONG_ONLY_MODE:
            sig = np.where(bt['proba'].values < bt['thr_short'].values, -1, sig)
        weak = (np.abs(bt['proba'].values - 0.5) < bt['edge'].values)
        sig[weak] = 0
    bt['raw_signal'] = sig
    bt['weight'] = 0.0
    gross_side = 1.0 if LONG_ONLY_MODE else 0.5
    for side in ([1] if LONG_ONLY_MODE else [1, -1]):
        c = bt[bt['raw_signal'] == side]
        if c.empty:
            continue
        rank = c.groupby('date')['_score'].rank(ascending=(side == -1), method='first')
        c = c[rank <= top_k]
        inv = 1.0 / c['vol_sizing'].clip(lower=1e-4)
        w = (inv / inv.groupby(c['date']).transform('sum') * gross_side).clip(upper=max_weight)
        bt.loc[c.index, 'weight'] = side * w
    # NB: a missing VIX percentile (only the first ~252 days of market history, long before MODEL_START) defaults to 0.0,
    # i.e. FULL exposure rather than reduced exposure. Inert here because no model row falls in that window.
    p = bt['vix_pctile'].fillna(0.0)
    scale = (1.0 - (p - soft) / (hard - soft)).clip(lower=min_scale, upper=1.0)
    bt['weight'] = bt['weight'] * scale
    bt['position'] = np.sign(bt['weight'])
    return bt.drop(columns='_score')

def apply_drawdown_breaker(return_series, dd_limit=DRAWDOWN_LIMIT, dd_recovery=DRAWDOWN_RECOVERY,
                           scale_during_breach=DRAWDOWN_SCALE):
    """Lookahead-free whole-book circuit breaker. The on/off state is decided on the shadow
    (unscaled) equity curve of the signal, using returns up to the previous period only.
    Deciding re-entry on the realised curve would freeze it during a breach (scale 0), so the
    book would stay flat for good. Here the book re-enters once the signal itself has recovered."""
    returns = return_series.copy().sort_index()
    shadow, peak, in_breach, out = 1.0, 1.0, False, []
    for r in returns.values:
        out.append(r * scale_during_breach if in_breach else r)
        shadow *= (1 + r); peak = max(peak, shadow)
        dd = 1 - shadow / peak
        if not in_breach and dd >= dd_limit:
            in_breach = True
        elif in_breach and dd <= dd_recovery:
            in_breach = False
    return pd.Series(out, index=returns.index, name=returns.name)

def select_rebalance_rows(bt, horizon):
    all_dates = np.sort(bt['date'].unique())
    return bt[bt['date'].isin(set(all_dates[::horizon]))].reset_index(drop=True)

def compute_edge_cutoff(proba, percentile=MIN_SIGNAL_PERCENTILE):
    return float(np.quantile(np.abs(np.asarray(proba) - 0.5), percentile))

def _quick_return_score(proba, meta_df, tl, ts, horizon, edge):
    """Grid objective: compounded return of the pooled book on the selection window.
    Weights are summed per date first."""
    bt = meta_df.copy(); bt['proba'] = proba
    bt['thr_long'], bt['thr_short'], bt['edge'] = tl, ts, edge
    bt = build_position_weights(select_rebalance_rows(bt, horizon))
    port = (bt['weight'] * bt['fwd_return']).groupby(bt['date']).sum()
    if (port != 0).sum() < MIN_RELIABLE_TRADES:
        return -np.inf
    return (1 + port).prod() - 1

def select_asymmetric_thresholds(proba, meta_df, horizon, long_grid=None, short_grid=None):
    long_grid = long_grid or [0.52, 0.55, 0.58, 0.60, 0.65, 0.70]
    short_grid = short_grid or [0.48, 0.45, 0.42, 0.40, 0.35, 0.30]
    eff_short = [short_grid[0]] if LONG_ONLY_MODE else short_grid
    edge = compute_edge_cutoff(proba)
    if PORTFOLIO_MODE == 'rank':      # thresholds are not read in rank mode (build_position_weights): skip the grid search
        return float(long_grid[0]), float(eff_short[0]), pd.DataFrame(), edge
    rows = [dict(threshold_long=tl, threshold_short=ts,
                 return_score=_quick_return_score(proba, meta_df, tl, ts, horizon, edge))
            for tl in long_grid for ts in eff_short]
    grid = pd.DataFrame(rows).sort_values('return_score', ascending=False, kind='stable')
    best = grid.iloc[0]
    return float(best['threshold_long']), float(best['threshold_short']), grid, edge

def assign_rolling_thresholds(proba, meta, init_tl, init_ts, init_edge, horizon,
                              lookback_days=THRESHOLD_LOOKBACK_DAYS, min_days=THRESHOLD_MIN_DAYS):
    """Adds thr_long / thr_short / edge columns to meta. For each retrain block, thresholds are
    selected on the previous `lookback_days` of OOS predictions whose labels are fully realised
    (dates < block_start - horizon); the first block(s) fall back to the initial validation choice."""
    m = meta.copy(); m['proba'] = proba
    all_dates = np.sort(m['date'].unique())
    m['thr_long'], m['thr_short'], m['edge'] = init_tl, init_ts, init_edge
    for bs in np.sort(m['block_start'].unique()):
        pos = int(np.searchsorted(all_dates, bs))
        hist_end = pos - horizon
        hist_start = max(0, hist_end - lookback_days)
        if hist_end - hist_start < min_days:
            continue
        win = m[(m['date'] >= all_dates[hist_start]) & (m['date'] < all_dates[hist_end])]
        tl, ts, _, edge = select_asymmetric_thresholds(win['proba'].values, win, horizon)
        m.loc[m['block_start'] == bs, ['thr_long', 'thr_short', 'edge']] = [tl, ts, edge]
    return m.drop(columns=['proba'])

# ---------------------------------------------------------------------------
# Backtest and metrics
# ---------------------------------------------------------------------------
# Risk-free return per rebalance period: 3-month T-bill (FRED DTB3, already lagged one business day in `market`),
# as a simple rate prorated to the HORIZON-day holding period. Used ONLY for `sharpe_excess`.
RF_PERIOD = market['tbill3m'] / 100.0 * HORIZON / 252.0

def perf_stats(r, rf=None):
    """`sharpe` is computed on raw period returns (risk-free = 0). If `rf` (a date-indexed
    per-period risk-free return, RF_PERIOD) is given, `sharpe_excess` uses r - rf with the rate in force on each date.
    Idle cash is not credited with interest, so sharpe_excess is slightly conservative for strategies whose gross
    exposure is below 100% (the benchmarks are fully invested, so for them it is exact)."""
    r = pd.Series(r).dropna()
    n = len(r)
    if n == 0:
        return dict(cumulative_return=np.nan, ann_return=np.nan, ann_vol=np.nan, sharpe=np.nan, sharpe_excess=np.nan,
                    max_drawdown=np.nan)
    cum = (1 + r).prod() - 1
    ann = (1 + cum) ** (PERIODS_PER_YEAR / n) - 1 if cum > -1 else -1.0
    vol = r.std() * np.sqrt(PERIODS_PER_YEAR)
    sharpe = r.mean() / r.std() * np.sqrt(PERIODS_PER_YEAR) if r.std() > 0 else np.nan
    sharpe_ex = np.nan
    if rf is not None:
        ex = r - rf.reindex(rf.index.union(r.index)).ffill().reindex(r.index).fillna(0.0)
        sharpe_ex = ex.mean() / ex.std() * np.sqrt(PERIODS_PER_YEAR) if ex.std() > 0 else np.nan
    eq = (1 + r).cumprod()
    return dict(cumulative_return=cum, ann_return=ann, ann_vol=vol, sharpe=sharpe, sharpe_excess=sharpe_ex,
                max_drawdown=float((eq / eq.cummax() - 1).min()))

def _activity(r):
    active = r[r != 0]
    n = len(active)
    if n < MIN_RELIABLE_TRADES:
        return np.nan, np.nan, n
    return (active > 0).mean(), active.mean(), n

def _window(meta, eval_start):
    return meta if eval_start is None else meta[meta['date'] >= pd.Timestamp(eval_start)]

def build_book(proba, meta_df, horizon, cost_bps=TRANSACTION_COST_BPS, eval_start=None):
    bt = meta_df.copy(); bt['proba'] = proba
    bt = _window(bt, eval_start).sort_values(['ticker', 'date'])
    bt = build_position_weights(select_rebalance_rows(bt, horizon))
    bt = bt.sort_values(['ticker', 'date'])
    bt['prev_weight'] = bt.groupby('ticker')['weight'].shift(1).fillna(0.0)
    bt['cost'] = (bt['weight'] - bt['prev_weight']).abs() * cost_bps / 1e4
    bt['strategy_return'] = bt['weight'] * bt['fwd_return'] - bt['cost']
    return bt

def pooled_series(bt, apply_breaker=True):
    port = bt.groupby('date').agg(strategy_return=('strategy_return', 'sum'),
                                  gross_exposure=('weight', lambda w: w.abs().sum()),
                                  n_positions=('weight', lambda w: int((w != 0).sum())),
                                  ew_buy_hold=('fwd_return', 'mean'),
                                  sp500=('sp500_fwd_return', 'first')).sort_index()
    port['strategy_raw'] = port['strategy_return']
    if apply_breaker:
        port['strategy_return'] = apply_drawdown_breaker(port['strategy_return'])
    return port

def run_backtest(proba, meta_df, horizon, cost_bps=TRANSACTION_COST_BPS, eval_start=None, apply_breaker=True):
    bt = build_book(proba, meta_df, horizon, cost_bps, eval_start)
    rows = []
    for t, g in bt.groupby('ticker'):
        r = g.set_index('date')['strategy_return']       # contribution of this name to the book
        hit, mar, n_act = _activity(r)
        rows.append(dict(ticker=t, series='strategy', **perf_stats(r), hit_rate=hit,
                         mean_active_return=mar, n_active=n_act, n_periods=len(r)))
    port = pooled_series(bt, apply_breaker)
    for col, label in [('strategy_return', 'strategy'), ('ew_buy_hold', 'ew_buy_hold'), ('sp500', 'sp500')]:
        hit, mar, n_act = _activity(port[col])
        rows.append(dict(ticker='POOLED', series=label, **perf_stats(port[col], rf=RF_PERIOD), hit_rate=hit,
                         mean_active_return=mar, n_active=n_act, n_periods=len(port),
                         avg_gross_exposure=port['gross_exposure'].mean() if label == 'strategy' else 1.0,
                         turnover_per_period=(bt['weight'] - bt['prev_weight']).abs().groupby(bt['date']).sum().mean()
                         if label == 'strategy' else np.nan))
    return pd.DataFrame(rows), {'POOLED': port}

def compute_pooled_return_series(proba, meta_df, horizon, cost_bps=TRANSACTION_COST_BPS,
                                 exclude_tickers=None, eval_start=None, apply_breaker=True):
    m = meta_df if not exclude_tickers else meta_df[~meta_df['ticker'].isin(exclude_tickers)]
    p = proba if not exclude_tickers else proba[~meta_df['ticker'].isin(exclude_tickers).values]
    return pooled_series(build_book(p, m, horizon, cost_bps, eval_start), apply_breaker)['strategy_return']

def compute_classification_metrics(proba, meta_df, by_ticker=True, eval_start=None):
    mask = np.ones(len(meta_df), bool) if eval_start is None else (meta_df['date'] >= pd.Timestamp(eval_start)).values
    meta_df, proba = meta_df[mask], proba[mask]
    actual = meta_df['target'].values; pred = (proba >= 0.5).astype(int)
    def _m(a, p):
        if len(a) == 0 or len(np.unique(a)) < 2:
            return dict(accuracy=np.nan, macro_f1=np.nan, mcc=np.nan, n=len(a))
        pr, rc, f1, sup = precision_recall_fscore_support(a, p, labels=[0, 1], zero_division=0)
        return dict(accuracy=accuracy_score(a, p), macro_f1=f1_score(a, p, average='macro'),
                    mcc=matthews_corrcoef(a, p), n=len(a),
                    down_precision=pr[0], down_recall=rc[0], down_f1=f1[0], down_support=sup[0],
                    up_precision=pr[1], up_recall=rc[1], up_f1=f1[1], up_support=sup[1])
    rows = [dict(ticker='POOLED', **_m(actual, pred))]
    if by_ticker:
        for t in meta_df['ticker'].unique():
            mk_ = (meta_df['ticker'] == t).values
            rows.append(dict(ticker=t, **_m(actual[mk_], pred[mk_])))
    return pd.DataFrame(rows)

def signal_ic(proba, meta_df, horizon, eval_start=None):
    m = meta_df.copy(); m['proba'] = proba
    m = select_rebalance_rows(_window(m, eval_start), horizon)
    ics = daily_rank_ic(m, 'proba', 'fwd_return')
    return dict(rank_ic=ics.mean(), rank_ic_t_hac=hac_tstat(ics), ic_hit_rate=(ics > 0).mean(), n_dates=len(ics))

def _format_backtest_row(label, r):
    n_active = int(r['n_active']) if pd.notna(r['n_active']) else 0
    base = f"  {label}: return={r['cumulative_return'] * 100:.2f}% | Sharpe={r['sharpe']:.2f} | Sharpe(excess of T-bill)={r['sharpe_excess']:.2f} | maxDD={r['max_drawdown'] * 100:.1f}%"
    if n_active < MIN_RELIABLE_TRADES:
        return base + f" | INSUFFICIENT DATA for hit rate ({n_active} active)"
    return base + (f" | hit rate={r['hit_rate'] * 100:.1f}% | mean/active={r['mean_active_return'] * 100:.3f}% "
                   f"| active={n_active}/{int(r['n_periods'])}")

def print_model_report(name, params, meta_thr, class_df, bt_df, ic_by_window, scope_tickers):
    print(f"\n{'=' * 78}\n  {name}\n{'=' * 78}")
    print(f"  hyperparameters: {params}")
    thr = meta_thr.groupby('block_start')[['thr_long', 'edge']].first()
    if PORTFOLIO_MODE == 'rank':
        print(f"  portfolio: top-{PORTFOLIO_TOP_K} ranking (thresholds not used)")
    else:
      print(f"  thresholds across {len(thr)} retrain blocks: long in [{thr.thr_long.min():.2f}, {thr.thr_long.max():.2f}] "
            f"(median {thr.thr_long.median():.2f}); conviction edge median {thr.edge.median():.3f}")
    for w, ic in ic_by_window.items():
        print(f"  [{w}] rank IC={ic['rank_ic']:.4f} (HAC t={ic['rank_ic_t_hac']:.2f}, IC>0 on {ic['ic_hit_rate']:.0%} of {ic['n_dates']} dates)")
    for w in bt_df['window'].unique():
        b = bt_df[bt_df['window'] == w]
        print(f"  --- backtest, window={w} ---")
        for series in ['strategy', 'ew_buy_hold', 'sp500']:
            row = b[(b.ticker == 'POOLED') & (b.series == series)]
            if len(row):
                print(_format_backtest_row(f"POOLED {series:<11}", row.iloc[0]))
    c = class_df[(class_df['window'] == 'full_oos') & (class_df['ticker'] == 'POOLED')].iloc[0]
    print(f"  classification (full OOS): macro F1={c['macro_f1']:.3f}, MCC={c['mcc']:.3f}, n={int(c['n'])}")

## 8. Model-family adapters

**a) Ridge (L2) logistic regression**

In [ ]:
import optuna
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
optuna.logging.set_verbosity(optuna.logging.WARNING)

def logreg_tune(train_df, feature_cols, n_trials=10, n_splits=3):
    X, y = train_df[feature_cols].values, train_df['target'].values
    dates, fwd = train_df['date'].values, train_df['fwd_return'].values
    folds = list(purged_date_folds(dates, n_splits))
    def objective(trial):
        C = trial.suggest_float('C', 1e-4, 1e1, log=True)
        scores = []
        for tr, va in folds:
            if len(np.unique(y[tr])) < 2 or len(np.unique(y[va])) < 2:
                continue
            sc = StandardScaler().fit(X[tr])
            clf = LogisticRegression(C=C, class_weight='balanced', max_iter=2000, random_state=RANDOM_SEED)
            clf.fit(sc.transform(X[tr]), y[tr])
            p = clf.predict_proba(sc.transform(X[va]))[:, 1]
            scores.append(tuning_score(y[va], p, dates[va], fwd[va]))
        return np.mean(scores) if scores else 0.0
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params

def logreg_fit_predict(train_df, test_df, feature_cols, params):
    sc = StandardScaler()
    X_tr = sc.fit_transform(train_df[feature_cols].values); X_te = sc.transform(test_df[feature_cols].values)
    clf = LogisticRegression(C=params['C'], class_weight='balanced', max_iter=2000, random_state=RANDOM_SEED)
    clf.fit(X_tr, train_df['target'].values)
    return clf.predict_proba(X_te)[:, 1]

**b) XGBoost**

In [ ]:
import xgboost as xgb
XGB_DIAG = []   # filled by xgb_fit_predict; reported per feature set in Section 11

def xgb_tune(train_df, feature_cols, n_trials=10, n_splits=3):
    X, y = train_df[feature_cols].values, train_df['target'].values
    dates, fwd = train_df['date'].values, train_df['fwd_return'].values
    folds = list(purged_date_folds(dates, n_splits))
    pos_weight = (y == 0).sum() / max((y == 1).sum(), 1)
    def objective(trial):
        params = dict(max_depth=trial.suggest_int('max_depth', 2, 5),
                      learning_rate=trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
                      n_estimators=trial.suggest_int('n_estimators', 50, 400),
                      subsample=trial.suggest_float('subsample', 0.5, 1.0),
                      colsample_bytree=trial.suggest_float('colsample_bytree', 0.3, 1.0),
                      min_child_weight=trial.suggest_int('min_child_weight', 5, 50),
                      reg_lambda=trial.suggest_float('reg_lambda', 1e-2, 20.0, log=True))
        scores = []
        for tr, va in folds:
            if len(np.unique(y[tr])) < 2 or len(np.unique(y[va])) < 2:
                continue
            clf = xgb.XGBClassifier(**params, scale_pos_weight=pos_weight, random_state=RANDOM_SEED,
                                    eval_metric='logloss', verbosity=0, n_jobs=1,
                                    early_stopping_rounds=EARLY_STOPPING_ROUNDS)
            clf.fit(X[tr], y[tr], eval_set=[(X[va], y[va])], verbose=False)
            scores.append(tuning_score(y[va], clf.predict_proba(X[va])[:, 1], dates[va], fwd[va]))
        return np.mean(scores) if scores else 0.0
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params

def xgb_fit_predict(train_df, test_df, feature_cols, params):
    """Phase A: early stopping on a chronological, purged inner split finds the number of trees.
    Phase B: refit on the full training window for that many trees."""
    tr_dates = np.sort(train_df['date'].unique())
    split = min(max(int(len(tr_dates) * (1 - EARLY_STOP_VAL_FRAC)), PURGE_DAYS + 1), len(tr_dates) - 1)
    inner_tr = train_df[train_df['date'] < tr_dates[split - PURGE_DAYS]]
    inner_ev = train_df[train_df['date'] >= tr_dates[split]]
    X_te = test_df[feature_cols].values
    X_full, y_full = train_df[feature_cols].values, train_df['target'].values
    pw_full = (y_full == 0).sum() / max((y_full == 1).sum(), 1)
    n_trees = params['n_estimators']
    if inner_tr['target'].nunique() == 2 and inner_ev['target'].nunique() == 2:
        y_i = inner_tr['target'].values
        probe = xgb.XGBClassifier(**params, scale_pos_weight=(y_i == 0).sum() / max((y_i == 1).sum(), 1),
                                  random_state=RANDOM_SEED, eval_metric='logloss', verbosity=0, n_jobs=1,
                                  early_stopping_rounds=EARLY_STOPPING_ROUNDS)
        probe.fit(inner_tr[feature_cols].values, y_i,
                  eval_set=[(inner_ev[feature_cols].values, inner_ev['target'].values)], verbose=False)
        n_trees = probe.best_iteration + 1
    final = xgb.XGBClassifier(**{**params, 'n_estimators': n_trees}, scale_pos_weight=pw_full,
                              random_state=RANDOM_SEED, eval_metric='logloss', verbosity=0, n_jobs=1)
    final.fit(X_full, y_full)
    # diagnostics: tree count chosen by early stopping and the share of split gain on sentiment inputs
    gain = final.get_booster().get_score(importance_type='total_gain')
    names = {f'f{i}': c for i, c in enumerate(feature_cols)}
    tot = sum(gain.values()) or 1.0
    sent_share = sum(v for k, v in gain.items() if 'sentiment' in names.get(k, '') or 'has_news' in names.get(k, '')) / tot
    XGB_DIAG.append(dict(n_trees=n_trees, sentiment_gain_share=sent_share,
                         train_end=pd.Timestamp(train_df['date'].max())))
    return final.predict_proba(X_te)[:, 1]

**c) LSTM**

The sequence length `SEQ_LEN` comes from Section 3. The tuning split is date-based and purged, and the walk-forward and early-stopping splits carry the same label buffer as the tabular models.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def build_sequences(df, feature_cols, seq_len):
    d = df[df.ticker.isin(TICKERS)].dropna(subset=feature_cols + ['target', 'fwd_return', 'vol_sizing'])
    tickers = sorted(d['ticker'].unique()); t2i = {t: i for i, t in enumerate(tickers)}
    X, y, tidx, metas = [], [], [], []
    for t, g in d.groupby('ticker', sort=False):
        g = g.sort_values('date').reset_index(drop=True)
        feats = g[feature_cols].values.astype(np.float32)
        if len(g) < seq_len:
            continue
        win = np.lib.stride_tricks.sliding_window_view(feats, (seq_len, feats.shape[1]))[:, 0]
        X.append(win); y.append(g['target'].values[seq_len - 1:])
        tidx.append(np.full(len(win), t2i[t])); metas.append(g.loc[seq_len - 1:, META_COLS])
    X = np.concatenate(X).astype(np.float32); y = np.concatenate(y).astype(np.float32)
    tidx = np.concatenate(tidx).astype(np.int64)
    meta = pd.concat(metas, ignore_index=True)
    order = np.lexsort((meta['ticker'].values, meta['date'].values))   # chronological order
    return X[order], y[order], tidx[order], meta.iloc[order].reset_index(drop=True), t2i

def _scale_sequences(X_train, *X_others):
    from sklearn.preprocessing import StandardScaler
    nf = X_train.shape[-1]
    sc = StandardScaler().fit(X_train.reshape(-1, nf))
    tr = sc.transform(X_train.reshape(-1, nf)).reshape(X_train.shape).astype(np.float32)
    others = [Xo if len(Xo) == 0 else sc.transform(Xo.reshape(-1, nf)).reshape(Xo.shape).astype(np.float32) for Xo in X_others]
    return (tr, *others) if others else tr

class SeqDataset(Dataset):
    def __init__(self, X, y, tidx):
        self.X = torch.tensor(X, dtype=torch.float32); self.y = torch.tensor(y, dtype=torch.long)
        self.tidx = torch.tensor(tidx, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i], self.tidx[i]

class LSTMClassifier(nn.Module):
    def __init__(self, n_features, hidden_size, num_tickers, embed_dim=4, dropout=0.3):
        super().__init__()
        self.use_embedding = num_tickers > 1
        self.embedding = nn.Embedding(num_tickers, embed_dim) if self.use_embedding else None
        self.lstm = nn.LSTM(n_features, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size + (embed_dim if self.use_embedding else 0), 2)
    def forward(self, x, tidx):
        _, (h_n, _) = self.lstm(x)
        h = self.dropout(h_n[-1])
        if self.use_embedding:
            h = torch.cat([h, self.embedding(tidx)], dim=1)
        return self.fc(h)

def compute_class_weights(y):
    classes, counts = np.unique(np.asarray(y), return_counts=True)
    w = np.ones(2)
    for c, wt in zip(classes, len(y) / (len(classes) * counts)):
        w[int(c)] = wt
    return torch.tensor(w, dtype=torch.float32)

def _train_one(X_tr, y_tr, tidx_tr, n_features, num_tickers, params, epochs,
               X_val=None, y_val=None, tidx_val=None, patience=EARLY_STOP_PATIENCE):
    # seed EVERY training run: otherwise the initial weights depend on how many runs came before (e.g. on how many tuning
    # calls were skipped by SHARED_HYPERPARAMS), and results change with call order rather than with the features
    torch.manual_seed(RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_SEED)
    model = LSTMClassifier(n_features, params['hidden_size'], num_tickers, dropout=params['dropout']).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])
    loss_fn = nn.CrossEntropyLoss(weight=compute_class_weights(y_tr).to(DEVICE))
    g = torch.Generator(); g.manual_seed(RANDOM_SEED)
    dl = DataLoader(SeqDataset(X_tr, y_tr, tidx_tr), batch_size=params['batch_size'], shuffle=True, generator=g)
    use_es = X_val is not None and len(X_val) > 0
    best_loss, best_state, best_epoch, bad = np.inf, None, epochs, 0
    if use_es:
        Xv = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
        yv = torch.tensor(y_val, dtype=torch.long).to(DEVICE); tv = torch.tensor(tidx_val, dtype=torch.long).to(DEVICE)
    for epoch in range(epochs):
        model.train()
        for xb, yb, tb in dl:
            xb, yb, tb = xb.to(DEVICE), yb.to(DEVICE), tb.to(DEVICE)
            opt.zero_grad(); loss = loss_fn(model(xb, tb), yb); loss.backward(); opt.step()
        if use_es:
            model.eval()
            with torch.no_grad():
                vl = loss_fn(model(Xv, tv), yv).item()
            if vl < best_loss:
                best_loss, best_epoch, bad = vl, epoch + 1, 0
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= patience:
                    break
    if use_es and best_state is not None:
        model.load_state_dict(best_state)
        return model, best_epoch
    return model

def _predict(model, X_te, tidx_te, batch=4096):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X_te), batch):
            xb = torch.tensor(X_te[i:i + batch], dtype=torch.float32).to(DEVICE)
            tb = torch.tensor(tidx_te[i:i + batch], dtype=torch.long).to(DEVICE)
            out.append(torch.softmax(model(xb, tb), dim=1)[:, 1].cpu().numpy())
    return np.concatenate(out) if out else np.array([])

def _date_split_masks(meta_dates, frac, purge=PURGE_DAYS):
    u = np.sort(np.unique(meta_dates)); split = int(len(u) * frac)
    split = min(max(split, purge + 1), len(u) - 1)
    return (meta_dates < u[split - purge]), (meta_dates >= u[split])

def lstm_tune(X, y, tidx, meta_sub, n_features, num_tickers, n_trials=5, tuning_epochs=5, val_frac=0.2):
    tr_m, va_m = _date_split_masks(meta_sub['date'].values, 1 - val_frac)
    X_tr, X_va = _scale_sequences(X[tr_m], X[va_m])
    y_tr, y_va, t_tr, t_va = y[tr_m], y[va_m], tidx[tr_m], tidx[va_m]
    d_va, f_va = meta_sub['date'].values[va_m], meta_sub['fwd_return'].values[va_m]
    def objective(trial):
        params = dict(hidden_size=trial.suggest_categorical('hidden_size', [8, 16, 32]),
                      dropout=trial.suggest_float('dropout', 0.1, 0.5),
                      lr=trial.suggest_float('lr', 1e-4, 5e-3, log=True),
                      weight_decay=trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True),
                      batch_size=trial.suggest_categorical('batch_size', [64, 128]))
        if len(np.unique(y_tr)) < 2 or len(np.unique(y_va)) < 2:
            return 0.0
        model, _ = _train_one(X_tr, y_tr, t_tr, n_features, num_tickers, params, tuning_epochs,
                              X_val=X_va, y_val=y_va, tidx_val=t_va)
        return tuning_score(y_va, _predict(model, X_va, t_va), d_va, f_va)
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params

def walk_forward_lstm(X, y, tidx, meta, n_features, num_tickers, params, initial_train_days,
                      retrain_step_days, epochs, purge=PURGE_DAYS, train_window_days=TRAIN_WINDOW_DAYS):
    dates = np.sort(meta['date'].unique()); md_ = meta['date'].values
    oos_proba, oos_meta = [], []
    start = initial_train_days
    while start < len(dates):
        end = min(start + retrain_step_days, len(dates))
        tr_end = start - purge
        tr_start = 0 if train_window_days is None else max(0, tr_end - train_window_days)
        train_mask = (md_ >= dates[tr_start]) & (md_ < dates[tr_end])
        test_mask = (md_ >= dates[start]) & (md_ <= dates[end - 1])
        if test_mask.sum() == 0 or len(np.unique(y[train_mask])) < 2:
            start = end; continue
        in_tr, in_va = _date_split_masks(md_[train_mask], 1 - EARLY_STOP_VAL_FRAC)
        idx_tr = np.where(train_mask)[0]
        inner_tr_mask = np.zeros(len(y), bool); inner_tr_mask[idx_tr[in_tr]] = True
        inner_va_mask = np.zeros(len(y), bool); inner_va_mask[idx_tr[in_va]] = True
        if inner_va_mask.sum() == 0 or len(np.unique(y[inner_va_mask])) < 2:
            Xtr_s, Xte_s = _scale_sequences(X[train_mask], X[test_mask])
            model = _train_one(Xtr_s, y[train_mask], tidx[train_mask], n_features, num_tickers, params, epochs)
        else:
            Xi_tr, Xi_va = _scale_sequences(X[inner_tr_mask], X[inner_va_mask])
            _, best_epoch = _train_one(Xi_tr, y[inner_tr_mask], tidx[inner_tr_mask], n_features, num_tickers,
                                       params, epochs, X_val=Xi_va, y_val=y[inner_va_mask], tidx_val=tidx[inner_va_mask])
            Xtr_s, Xte_s = _scale_sequences(X[train_mask], X[test_mask])
            model = _train_one(Xtr_s, y[train_mask], tidx[train_mask], n_features, num_tickers, params, epochs=best_epoch)
        oos_proba.append(_predict(model, Xte_s, tidx[test_mask]))
        m = meta.loc[test_mask, META_COLS].copy(); m['block_start'] = dates[start]
        oos_meta.append(m)
        start = end
    return np.concatenate(oos_proba), pd.concat(oos_meta, ignore_index=True)

## 9. Collapse detection and significance testing

In [ ]:
def bootstrap_paired_return_difference(returns1, returns2, block_size=None, n_boot=1000, ci=0.95, seed=None):
    block_size = BLOCK_SIZE if block_size is None else block_size
    seed = BOOT_SEED if seed is None else seed
    al = pd.concat([returns1.rename('r1'), returns2.rename('r2')], axis=1).fillna(0.0).sort_index()
    x1, x2 = al['r1'].values, al['r2'].values; n = len(x1)
    if n < 10:
        return dict(point=np.nan, ci_lower=np.nan, ci_upper=np.nan, cum_return_1=np.nan, cum_return_2=np.nan, n_periods=n)
    rng = np.random.default_rng(seed); max_start = max(n - block_size, 1); diffs = []
    for _ in range(n_boot):
        starts = rng.integers(0, max_start + 1, size=int(np.ceil(n / block_size)))
        idx = np.clip(np.concatenate([np.arange(s, s + block_size) for s in starts])[:n], 0, n - 1)
        diffs.append(((1 + x1[idx]).prod() - 1) - ((1 + x2[idx]).prod() - 1))
    c1, c2 = (1 + x1).prod() - 1, (1 + x2).prod() - 1
    return dict(point=c1 - c2, ci_lower=np.percentile(diffs, (1 - ci) / 2 * 100),
                ci_upper=np.percentile(diffs, (1 + ci) / 2 * 100), cum_return_1=c1, cum_return_2=c2, n_periods=n)

def permutation_test_return_difference(returns1, returns2, block_size=None, n_perm=2000, seed=None):
    block_size = BLOCK_SIZE if block_size is None else block_size
    seed = BOOT_SEED if seed is None else seed
    al = pd.concat([returns1.rename('r1'), returns2.rename('r2')], axis=1).fillna(0.0).sort_index()
    x1, x2 = al['r1'].values, al['r2'].values; n = len(x1)
    if n < 10:
        return dict(observed_diff=np.nan, p_value=np.nan, n_periods=n, n_perm=n_perm)
    obs = ((1 + x1).prod() - 1) - ((1 + x2).prod() - 1)
    nb = int(np.ceil(n / block_size)); bid = np.repeat(np.arange(nb), block_size)[:n]
    rng = np.random.default_rng(seed); perm = np.empty(n_perm)
    for k in range(n_perm):
        flip = rng.integers(0, 2, size=nb).astype(bool)[bid]
        perm[k] = ((1 + np.where(flip, x2, x1)).prod() - 1) - ((1 + np.where(flip, x1, x2)).prod() - 1)
    return dict(observed_diff=obs, p_value=(np.sum(np.abs(perm) >= abs(obs)) + 1) / (n_perm + 1), n_periods=n, n_perm=n_perm)

def p_value_significance_label(p, alpha=0.05):
    if pd.isna(p):
        return 'not testable (insufficient data)'
    return f'SIGNIFICANT (p={p:.3f} < {alpha})' if p < alpha else f'not significant (p={p:.3f})'

def detect_collapsed_tickers(proba_by_model, meta_by_model, std_epsilon=0.01, identical_corr=0.999, identical_diff=0.005):
    names = list(proba_by_model); tickers = sorted(meta_by_model[names[0]]['ticker'].unique())
    flags, reasons = {}, {}
    for t in tickers:
        reason, masks = [], {}
        for nm in names:
            masks[nm] = (meta_by_model[nm]['ticker'] == t).values
            if masks[nm].sum() and proba_by_model[nm][masks[nm]].std() < std_epsilon:
                reason.append(f'{nm} proba std={proba_by_model[nm][masks[nm]].std():.4f} (near-constant)')
        for i in range(len(names)):
            for j in range(i + 1, len(names)):
                pa, pb = proba_by_model[names[i]][masks[names[i]]], proba_by_model[names[j]][masks[names[j]]]
                if len(pa) == len(pb) and len(pa) > 5 and pa.std() > 0 and pb.std() > 0:
                    corr, diff = np.corrcoef(pa, pb)[0, 1], np.abs(pa - pb).mean()
                    if corr > identical_corr and diff < identical_diff:
                        reason.append(f'{names[i]} & {names[j]} near-identical (corr={corr:.4f})')
        flags[t], reasons[t] = bool(reason), reason
    return flags, reasons

def significance_report(proba_by_model, meta_by_model, horizon=HORIZON, eval_start=SENTIMENT_EVAL_START, verbose=True, apply_breaker=True):
    sub_p, sub_m = {}, {}
    for nm in proba_by_model:
        mask = (meta_by_model[nm]['date'] >= pd.Timestamp(eval_start)).values
        sub_p[nm], sub_m[nm] = proba_by_model[nm][mask], meta_by_model[nm][mask].reset_index(drop=True)
    flags, reasons = detect_collapsed_tickers(sub_p, sub_m)
    collapsed = [t for t, f in flags.items() if f]
    ret, ret_ex, n_act = {}, {}, {}
    for nm in sub_p:
        ret[nm] = compute_pooled_return_series(sub_p[nm], sub_m[nm], horizon, apply_breaker=apply_breaker)
        ret_ex[nm] = compute_pooled_return_series(sub_p[nm], sub_m[nm], horizon, exclude_tickers=collapsed,
                                                  apply_breaker=apply_breaker)
        n_act[nm] = int((ret[nm] != 0).sum())
    rows = []
    for a, b in [('baseline', 'finbert'), ('baseline', 'qwen'), ('finbert', 'qwen')]:
        for subset, ra, rb in [('all_tickers', ret[a], ret[b]), ('excl_collapsed', ret_ex[a], ret_ex[b])]:
            ci = bootstrap_paired_return_difference(ra, rb); perm = permutation_test_return_difference(ra, rb)
            rows.append(dict(comparison=f'{a} vs {b}', subset=subset, window=f'from {eval_start}',
                             return_a=ci['cum_return_1'] * 100, return_b=ci['cum_return_2'] * 100,
                             diff=ci['point'] * 100, ci_lower=ci['ci_lower'] * 100, ci_upper=ci['ci_upper'] * 100,
                             p_value=perm['p_value'], significant=p_value_significance_label(perm['p_value'])))
    sig_df = pd.DataFrame(rows)
    if verbose:
        low = [nm for nm, n in n_act.items() if n < MIN_RELIABLE_TRADES]
        if low:
            print(f"\n INSUFFICIENT DATA: {', '.join(low)} traded fewer than {MIN_RELIABLE_TRADES} periods.")
        print(f"\nSentiment comparisons, rebalance dates from {eval_start}:")
        for _, r in sig_df.iterrows():
            print(f"[{r['subset']}] {r['comparison']}: {r['return_a']:.2f}% vs {r['return_b']:.2f}%, diff={r['diff']:.2f}% "
                  f"(95% CI [{r['ci_lower']:.2f}%, {r['ci_upper']:.2f}%]) -- {r['significant']}")
    return sig_df, flags, reasons

## 10. Orchestration

For each architecture: tune on the initial window (purged folds, by default on the baseline feature set only, see `SHARED_HYPERPARAMS`). Then, for each feature set: initial thresholds on a purged validation slice, purged walk-forward, rolling thresholds (inert in `'rank'` mode), and backtest and metrics on the evaluation window(s). An in-sample snapshot of the initial training window is kept for the overfitting check.

In [ ]:
# If sentiment exists from the first out-of-sample date (2015-2025 corpus) the 'sentiment era' IS the full OOS period: one
# window, not two identical copies of every table and test.
EVAL_WINDOWS = {'full_oos': None}
if pd.Timestamp(SENTIMENT_EVAL_START) > pd.Timestamp(FIRST_OOS_DATE):
    EVAL_WINDOWS['sentiment_era'] = SENTIMENT_EVAL_START

def _evaluate(proba, meta_thr, arch, fs_name):
    bts, cms, ics = [], [], {}
    for w, start in EVAL_WINDOWS.items():
        bt, _ = run_backtest(proba, meta_thr, HORIZON, eval_start=start)
        cm = compute_classification_metrics(proba, meta_thr, by_ticker=True, eval_start=start)
        ics[w] = signal_ic(proba, meta_thr, HORIZON, eval_start=start)
        for x in (bt, cm):
            x['window'] = w; x['feature_set'] = fs_name; x['architecture'] = arch
        bts.append(bt); cms.append(cm)
    ic_rows = [dict(architecture=arch, feature_set=fs_name, window=w, **v) for w, v in ics.items()]
    return pd.concat(bts), pd.concat(cms), ic_rows, ics

def run_tabular_family(df, architecture, tune_fn, fit_predict_fn, retrain_step_days, n_trials, verbose=True,
                       shared_params=None, feature_sets=None, share_hyperparams=None):
    """Hyperparameters: `shared_params` (if given) is used for every feature set; otherwise, when SHARED_HYPERPARAMS (or
    `share_hyperparams`) is on, the first feature set (baseline) is tuned and its settings are reused for the others; otherwise
    each feature set is tuned separately. `feature_sets` optionally restricts the run to a subset of FEATURE_SETS."""
    _share = SHARED_HYPERPARAMS if share_hyperparams is None else share_hyperparams
    _first_params = None
    bt_rows, cm_rows, ic_rows, cm_train_rows = [], [], [], []
    proba_by, meta_by, params_by = {}, {}, {}
    for fs_name, feature_cols in (feature_sets or FEATURE_SETS).items():
        d_pool, dummy_cols = add_ticker_dummies(prepare_tabular(df, feature_cols))
        cols = feature_cols + dummy_cols
        dates = np.sort(d_pool['date'].unique()); init = n_initial_days(dates)
        train_slice = d_pool[d_pool.date < dates[init - PURGE_DAYS]]
        if shared_params is not None:
            params = shared_params
        elif _share and _first_params is not None:
            params = _first_params                      # reuse the baseline's tuned settings
        else:
            params = tune_fn(train_slice, cols, n_trials=n_trials)
        if _first_params is None:
            _first_params = params
        params_by[fs_name] = params

        val_tr, val_df = get_validation_split(d_pool, init)
        val_proba = fit_predict_fn(val_tr, val_df, cols, params)
        tl0, ts0, _, edge0 = select_asymmetric_thresholds(val_proba, val_df, HORIZON)

        tr_proba = fit_predict_fn(train_slice, train_slice, cols, params)
        cmt = compute_classification_metrics(tr_proba, train_slice, by_ticker=False)
        cmt['feature_set'] = fs_name; cmt['architecture'] = architecture; cm_train_rows.append(cmt)

        if architecture == 'XGBoost':
            XGB_DIAG.clear()
        proba, meta = walk_forward(d_pool, cols, lambda tr, te, fc: fit_predict_fn(tr, te, fc, params),
                                   init, retrain_step_days)
        if architecture == 'XGBoost' and verbose and XGB_DIAG:
            dg = pd.DataFrame(XGB_DIAG)
            print(f"  [XGBoost diagnostics, {fs_name}] trees per walk-forward block: {dg.n_trees.tolist()}; "
                  f"sentiment share of split gain: {[round(x, 3) for x in dg.sentiment_gain_share]}")
            if dg.n_trees.median() < 20:
                print("  NOTE: early stopping keeps very few trees -> near-constant scores. This means the "
                      "validation log-loss finds little signal, not a code error; expect identical results "
                      "for feature sets whose extra inputs are never split on.")
        meta_thr = assign_rolling_thresholds(proba, meta, tl0, ts0, edge0, HORIZON)
        bt, cm, icr, ics = _evaluate(proba, meta_thr, architecture, fs_name)
        bt_rows.append(bt); cm_rows.append(cm); ic_rows += icr
        proba_by[fs_name], meta_by[fs_name] = proba, meta_thr
        if verbose:
            print_model_report(f"{architecture} | {fs_name} | POOLED ({len(cols)} inputs)", params, meta_thr, cm, bt, ics, TICKERS)
            print(f"  [train snapshot, initial window, in-sample] macro F1={cmt['macro_f1'].iloc[0]:.3f}, MCC={cmt['mcc'].iloc[0]:.3f}")
    return dict(bt=pd.concat(bt_rows, ignore_index=True), cm=pd.concat(cm_rows, ignore_index=True),
                ic=pd.DataFrame(ic_rows), cm_train=pd.concat(cm_train_rows, ignore_index=True),
                proba_by_model=proba_by, meta_by_model=meta_by, params_by_model=params_by)

def run_lstm_family(df, retrain_step_days, n_trials, tuning_epochs, epochs, verbose=True, shared_params=None, share_hyperparams=None):
    _share = SHARED_HYPERPARAMS if share_hyperparams is None else share_hyperparams
    _first_params = None
    bt_rows, cm_rows, ic_rows, cm_train_rows = [], [], [], []
    proba_by, meta_by, params_by = {}, {}, {}
    for fs_name, feature_cols in FEATURE_SETS.items():
        X, y, tidx, meta, t2i = build_sequences(df, feature_cols, SEQ_LEN)
        dates = np.sort(meta['date'].unique()); init = n_initial_days(dates)
        md_ = meta['date'].values
        train_mask = md_ < dates[init - PURGE_DAYS]
        if shared_params is not None:
            params = shared_params
        elif _share and _first_params is not None:
            params = _first_params                      # reuse the baseline's tuned settings
        else:
            params = lstm_tune(X[train_mask], y[train_mask], tidx[train_mask], meta[train_mask].reset_index(drop=True),
                               X.shape[2], len(t2i), n_trials=n_trials, tuning_epochs=tuning_epochs)
        if _first_params is None:
            _first_params = params
        params_by[fs_name] = params
        # initial thresholds: probe model trained on the purged first 80% of the initial window
        split = int(init * 0.8)
        vtr = md_ < dates[split - PURGE_DAYS]
        vva = (md_ >= dates[split]) & (md_ < dates[init - PURGE_DAYS])
        Xa, Xb = _scale_sequences(X[vtr], X[vva])
        probe, best_epoch = _train_one(Xa, y[vtr], tidx[vtr], X.shape[2], len(t2i), params, epochs,
                                       X_val=Xb, y_val=y[vva], tidx_val=tidx[vva])
        val_meta = meta[vva].reset_index(drop=True)
        tl0, ts0, _, edge0 = select_asymmetric_thresholds(_predict(probe, Xb, tidx[vva]), val_meta, HORIZON)

        Xfull = _scale_sequences(X[train_mask])
        snap = _train_one(Xfull, y[train_mask], tidx[train_mask], X.shape[2], len(t2i), params, epochs=best_epoch)
        cmt = compute_classification_metrics(_predict(snap, Xfull, tidx[train_mask]), meta[train_mask].reset_index(drop=True), by_ticker=False)
        cmt['feature_set'] = fs_name; cmt['architecture'] = 'LSTM'; cm_train_rows.append(cmt)

        proba, meta_oos = walk_forward_lstm(X, y, tidx, meta, X.shape[2], len(t2i), params, init, retrain_step_days, epochs)
        meta_thr = assign_rolling_thresholds(proba, meta_oos, tl0, ts0, edge0, HORIZON)
        bt, cm, icr, ics = _evaluate(proba, meta_thr, 'LSTM', fs_name)
        bt_rows.append(bt); cm_rows.append(cm); ic_rows += icr
        proba_by[fs_name], meta_by[fs_name] = proba, meta_thr
        if verbose:
            print_model_report(f"LSTM | {fs_name} | POOLED (seq_len={SEQ_LEN}, {X.shape[2]} inputs)", params, meta_thr, cm, bt, ics, TICKERS)
            print(f"  [train snapshot, initial window, in-sample] macro F1={cmt['macro_f1'].iloc[0]:.3f}, MCC={cmt['mcc'].iloc[0]:.3f}")
    return dict(bt=pd.concat(bt_rows, ignore_index=True), cm=pd.concat(cm_rows, ignore_index=True),
                ic=pd.DataFrame(ic_rows), cm_train=pd.concat(cm_train_rows, ignore_index=True),
                proba_by_model=proba_by, meta_by_model=meta_by, params_by_model=params_by)

# Protocol check: rows and dates are identical across feature sets, so every comparison is paired.
_tab = {fs: prepare_tabular(df, cols) for fs, cols in FEATURE_SETS.items()}
_rows = {fs: len(t) for fs, t in _tab.items()}
print(f"Rows per feature set: {_rows}")
assert len(set(_rows.values())) == 1, "feature sets must cover identical rows for paired tests"
# equal counts are not enough for PAIRED tests: the three sets must cover the very same (date, ticker) pairs
_ref = set(zip(_tab['baseline']['date'], _tab['baseline']['ticker']))
for _fs, _t in _tab.items():
    assert set(zip(_t['date'], _t['ticker'])) == _ref, f"{_fs} covers different (date, ticker) pairs than baseline"
print("Paired-sample check passed: all feature sets cover identical (date, ticker) pairs.")

## 11. Run all three families

In [ ]:
print("=" * 70); print("RUNNING: Ridge Logistic Regression"); print("=" * 70)
t0 = time.time()
results_logreg = run_tabular_family(df, 'Ridge LogReg', logreg_tune, logreg_fit_predict, RETRAIN_STEP_DAYS, N_TRIALS)
sig_logreg, flags_logreg, reasons_logreg = significance_report(results_logreg['proba_by_model'], results_logreg['meta_by_model'])
print(f"\nDone in {time.time() - t0:.1f}s. Collapsed tickers: {[t for t, f in flags_logreg.items() if f] or 'none'}")

In [ ]:
print("=" * 70); print("RUNNING: XGBoost"); print("=" * 70)
t0 = time.time()
results_xgb = run_tabular_family(df, 'XGBoost', xgb_tune, xgb_fit_predict, RETRAIN_STEP_DAYS, N_TRIALS)
sig_xgb, flags_xgb, reasons_xgb = significance_report(results_xgb['proba_by_model'], results_xgb['meta_by_model'])
print(f"\nDone in {time.time() - t0:.1f}s. Collapsed tickers: {[t for t, f in flags_xgb.items() if f] or 'none'}")

In [ ]:
print("=" * 70); print("RUNNING: LSTM"); print("=" * 70)
t0 = time.time()
results_lstm = run_lstm_family(df, RETRAIN_STEP_DAYS, N_TRIALS, LSTM_TUNING_EPOCHS, LSTM_EPOCHS)
sig_lstm, flags_lstm, reasons_lstm = significance_report(results_lstm['proba_by_model'], results_lstm['meta_by_model'])
print(f"\nDone in {time.time() - t0:.1f}s. Collapsed tickers: {[t for t, f in flags_lstm.items() if f] or 'none'}")

## 12. Cross-architecture comparison tables

In [ ]:
ALL_RESULTS = {'Ridge LogReg': results_logreg, 'XGBoost': results_xgb, 'LSTM': results_lstm}
all_bt = pd.concat([r['bt'] for r in ALL_RESULTS.values()], ignore_index=True)
all_cm = pd.concat([r['cm'] for r in ALL_RESULTS.values()], ignore_index=True)
all_ic = pd.concat([r['ic'] for r in ALL_RESULTS.values()], ignore_index=True)
all_cm_train = pd.concat([r['cm_train'] for r in ALL_RESULTS.values()], ignore_index=True)
all_bt.to_csv('three_family_backtest_results.csv', index=False)
all_cm.to_csv('three_family_classification_results.csv', index=False)
all_ic.to_csv('three_family_signal_ic.csv', index=False)
all_cm_train.to_csv('three_family_classification_results_train.csv', index=False)
FS_ORDER = ['baseline', 'finbert', 'qwen']

### 11b. Tie check: how often are the top-K picks decided by exact ties?

When early stopping keeps one or two trees, a model assigns the same score to several stocks. The book holds the top-K names by score, so exact ties at the cut-off are settled by a reproducible pseudo-random key (`_tie_key`) and not by row order, which would favour stocks early in the alphabet (AAPL, AMZN, GOOGL) over NVDA and TSLA. This table shows how often it matters: `% dates with a tie at the top-K cut-off` is the share of rebalance dates on which the K-th and (K+1)-th highest scores are identical, and `% dates all scores equal` is the extreme case.

In [ ]:
_rows = []
for _arch, _res in ALL_RESULTS.items():
    for _fs in FS_ORDER:
        _m = _res['meta_by_model'][_fs][['date', 'ticker']].copy(); _m['proba'] = _res['proba_by_model'][_fs]
        _m = select_rebalance_rows(_m, HORIZON)
        _sorted = _m.groupby('date')['proba'].apply(lambda p: np.sort(p.values)[::-1])
        _rows.append(dict(architecture=_arch, feature_set=_fs, rebalance_dates=len(_sorted),
                          **{'% dates with a tie at the top-K cut-off': 100 * np.mean([len(a) > PORTFOLIO_TOP_K and a[PORTFOLIO_TOP_K - 1] == a[PORTFOLIO_TOP_K] for a in _sorted])},
                          **{'% dates all scores equal': 100 * np.mean([len(a) > 1 and a[0] == a[-1] for a in _sorted])}))
tie_df = pd.DataFrame(_rows)
print(tie_df.round(1).to_string(index=False))
if tie_df['% dates with a tie at the top-K cut-off'].max() > 5:
    print("\nSome models decide a material share of their picks by ties: for those, portfolio returns are driven partly by the tie-break "
          "rather than by the model, so compare them with care.")

### 12a. Pooled strategy performance by window (returns, risk, benchmarks)

In [ ]:
pooled = all_bt[(all_bt.ticker == 'POOLED') & (all_bt.series == 'strategy')]
for w in EVAL_WINDOWS:
    sub = pooled[pooled.window == w]
    print(f"\n==================== window = {w} ====================")
    for metric, scale, fmt in [('cumulative_return', 100, 2), ('ann_return', 100, 2), ('sharpe', 1, 2), ('sharpe_excess', 1, 2),
                               ('max_drawdown', 100, 1), ('hit_rate', 100, 1), ('avg_gross_exposure', 100, 0)]:
        piv = sub.pivot_table(index='architecture', columns='feature_set', values=metric)[FS_ORDER] * scale
        print(f"\n{metric}{' (%)' if scale == 100 else ''}:"); print(piv.round(fmt))
    bench = all_bt[(all_bt.window == w) & (all_bt.ticker == 'POOLED') & (all_bt.series.isin(['ew_buy_hold', 'sp500']))]
    bench = bench.groupby('series')[['cumulative_return', 'ann_return', 'sharpe', 'sharpe_excess', 'max_drawdown']].first()
    print("\nBenchmarks on the same rebalance dates:"); print((bench * [100, 100, 1, 1, 100]).round(2))

### 12b. Signal quality: out-of-sample rank IC

In [ ]:
for w in EVAL_WINDOWS:
    sub = all_ic[all_ic.window == w]
    print(f"\nwindow = {w}: mean rank IC (HAC t-stat)")
    ic_p = sub.pivot_table(index='architecture', columns='feature_set', values='rank_ic')[FS_ORDER]
    t_p = sub.pivot_table(index='architecture', columns='feature_set', values='rank_ic_t_hac')[FS_ORDER]
    print(ic_p.round(4).astype(str) + ' (' + t_p.round(2).astype(str) + ')')

### 12c. Bootstrap confidence intervals for pooled cumulative return

In [ ]:
CI_LEVEL, CI_N_BOOT = 0.95, 1000
def bootstrap_cumulative_return_ci(r, block_size=BLOCK_SIZE, n_boot=CI_N_BOOT, ci=CI_LEVEL, seed=BOOT_SEED):
    x = r.values; n = len(x); rng = np.random.default_rng(seed); max_start = max(n - block_size, 1); out = []
    for _ in range(n_boot):
        starts = rng.integers(0, max_start + 1, size=int(np.ceil(n / block_size)))
        out.append((1 + np.concatenate([x[s:s + block_size] for s in starts])[:n]).prod() - 1)
    return (1 + x).prod() - 1, np.percentile(out, (1 - ci) / 2 * 100), np.percentile(out, (1 + ci) / 2 * 100)

ci_rows = []
for arch, res in ALL_RESULTS.items():
    for fs in FS_ORDER:
        for w, start in EVAL_WINDOWS.items():
            r = compute_pooled_return_series(res['proba_by_model'][fs], res['meta_by_model'][fs], HORIZON, eval_start=start)
            pt, lo, hi = bootstrap_cumulative_return_ci(r)
            ci_rows.append(dict(architecture=arch, feature_set=fs, window=w, return_point=pt * 100,
                                ci_lower=lo * 100, ci_upper=hi * 100, excludes_zero=bool(lo > 0 or hi < 0), n_periods=len(r)))
return_ci_df = pd.DataFrame(ci_rows)
print(return_ci_df.round(2).to_string(index=False))
return_ci_df.to_csv('return_bootstrap_confidence_intervals.csv', index=False)

### 12d. Per-stock contribution and classification (full OOS)

In [ ]:
per_stock = all_bt[(all_bt.ticker != 'POOLED') & (all_bt.window == 'full_oos')]
print("Cumulative contribution to the pooled book (%), by stock:")
print((per_stock.pivot_table(index=['architecture', 'ticker'], columns='feature_set', values='cumulative_return')[FS_ORDER] * 100).round(2))
f1 = all_cm[(all_cm.ticker != 'POOLED') & (all_cm.window == 'full_oos')]
print("\nMacro F1 by stock:")
print(f1.pivot_table(index=['architecture', 'ticker'], columns='feature_set', values='macro_f1')[FS_ORDER].round(3))

### 12e. Train (in-sample) vs. test (out-of-sample) -- overfitting check

In [ ]:
tvt = all_cm_train[['architecture', 'feature_set', 'macro_f1', 'mcc']].rename(columns={'macro_f1': 'train_macro_f1', 'mcc': 'train_mcc'}).merge(
    all_cm[(all_cm.ticker == 'POOLED') & (all_cm.window == 'full_oos')][['architecture', 'feature_set', 'macro_f1', 'mcc']].rename(
        columns={'macro_f1': 'test_macro_f1', 'mcc': 'test_mcc'}), on=['architecture', 'feature_set'])
tvt['macro_f1_gap'] = tvt['train_macro_f1'] - tvt['test_macro_f1']
tvt['mcc_gap'] = tvt['train_mcc'] - tvt['test_mcc']
print(tvt.round(3).to_string(index=False)); tvt.to_csv('train_vs_test_overfitting_check.csv', index=False)

### 12f. Per-class breakdown (pooled, full OOS)

In [ ]:
cols = ['architecture', 'feature_set', 'down_precision', 'down_recall', 'down_f1', 'up_precision', 'up_recall', 'up_f1']
print(all_cm[(all_cm.ticker == 'POOLED') & (all_cm.window == 'full_oos')][cols].round(3).to_string(index=False))

## 13. Sentiment significance tests, all architectures (from `SENTIMENT_EVAL_START`)

In [ ]:
sig_all = pd.concat([sig_logreg.assign(architecture='Ridge LogReg'), sig_xgb.assign(architecture='XGBoost'),
                     sig_lstm.assign(architecture='LSTM')], ignore_index=True)
sig_all.to_csv('three_family_significance_tests.csv', index=False)
for name, flags in [('Ridge LogReg', flags_logreg), ('XGBoost', flags_xgb), ('LSTM', flags_lstm)]:
    print(f"  {name}: collapsed = {[t for t, f in flags.items() if f] or 'none'}")
print(sig_all[['architecture', 'comparison', 'subset', 'return_a', 'return_b', 'diff', 'ci_lower', 'ci_upper',
               'p_value', 'significant']].round(3).to_string(index=False))

### 13b. Robustness: do the significance conclusions depend on the drawdown circuit breaker?

The breaker is path-dependent: whether a 15% drawdown is hit depends on the whole trajectory, so part of a return gap between two feature sets can come from the breaker reacting differently and not from the signal. The tests in Section 13 use the breaker (the same book that is reported). Below they are repeated **without** it (`apply_breaker=False`). If `same_conclusion` is True everywhere, the result does not depend on the breaker.

In [ ]:
_rob = []
for _arch, _res, _sig in [('Ridge LogReg', results_logreg, sig_logreg), ('XGBoost', results_xgb, sig_xgb), ('LSTM', results_lstm, sig_lstm)]:
    _nb_df, _, _ = significance_report(_res['proba_by_model'], _res['meta_by_model'], verbose=False, apply_breaker=False)
    _m = _sig.merge(_nb_df, on=['comparison', 'subset'], suffixes=('_breaker', '_nobreaker'))
    _m.insert(0, 'architecture', _arch)
    _rob.append(_m[['architecture', 'comparison', 'subset', 'diff_breaker', 'p_value_breaker', 'diff_nobreaker', 'p_value_nobreaker']])
rob_df = pd.concat(_rob, ignore_index=True)
rob_df['same_conclusion'] = (rob_df['p_value_breaker'] < 0.05) == (rob_df['p_value_nobreaker'] < 0.05)
rob_df.to_csv('three_family_significance_robustness_no_breaker.csv', index=False)
print("Sentiment-era return difference (percentage points) and permutation p-value, with vs without the drawdown breaker:")
print(rob_df.round(3).to_string(index=False))
print(f"\nConclusion (significant at 5% or not) unchanged in {int(rob_df['same_conclusion'].sum())}/{len(rob_df)} comparisons.")

### 13c. Sensitivity (optional): separate tuning per feature set

The main run uses one set of hyperparameters per model family (`SHARED_HYPERPARAMS = True`), so the three models differ only in their sentiment inputs. Set `RUN_OWN_TUNING_SENSITIVITY = True` to re-run XGBoost's +FinBERT and +Qwen with their own tuning and compare. This is not used for the reported results, and it re-tunes twice, so it takes several minutes.

In [ ]:
RUN_OWN_TUNING_SENSITIVITY = False
if RUN_OWN_TUNING_SENSITIVITY:
    own = run_tabular_family(df, 'XGBoost', xgb_tune, xgb_fit_predict, RETRAIN_STEP_DAYS, N_TRIALS, verbose=False,
                             feature_sets={k: FEATURE_SETS[k] for k in ['finbert', 'qwen']}, share_hyperparams=False)
    print("XGBoost hyperparameters: shared (main run) vs own tuning")
    for _k in ['finbert', 'qwen']:
        print(f"  {_k}: shared={results_xgb['params_by_model'][_k]}\n  {' ' * len(_k)}  own   ={own['params_by_model'][_k]}")
    _rows = []
    for _w in EVAL_WINDOWS:
        for _fs in FS_ORDER:
            _m = all_bt[(all_bt.architecture == 'XGBoost') & (all_bt.window == _w) & (all_bt.feature_set == _fs) & (all_bt.ticker == 'POOLED') & (all_bt.series == 'strategy')]
            _o = own['bt'][(own['bt'].window == _w) & (own['bt'].feature_set == _fs) & (own['bt'].ticker == 'POOLED') & (own['bt'].series == 'strategy')] if _fs != 'baseline' else _m
            _rows.append(dict(window=_w, feature_set=_fs, return_shared_pct=_m['cumulative_return'].iloc[0] * 100, return_own_tuning_pct=_o['cumulative_return'].iloc[0] * 100,
                              sharpe_shared=_m['sharpe'].iloc[0], sharpe_own_tuning=_o['sharpe'].iloc[0]))
    print("\nXGBoost strategy: shared hyperparameters (main) vs each feature set tuned on its own")
    print(pd.DataFrame(_rows).round(2).to_string(index=False))
else:
    print("Skipped (set RUN_OWN_TUNING_SENSITIVITY = True to run).")

## 14. Plots

In [ ]:
fig, axes = plt.subplots(1, len(ALL_RESULTS), figsize=(6 * len(ALL_RESULTS), 4.5), sharey=False, squeeze=False)
for ax, (arch, res) in zip(axes[0], ALL_RESULTS.items()):
    for fs in FS_ORDER:
        port = pooled_series(build_book(res['proba_by_model'][fs], res['meta_by_model'][fs], HORIZON))
        ax.plot(port.index, (1 + port['strategy_return']).cumprod(), label=fs, lw=1.2)
    ax.plot(port.index, (1 + port['ew_buy_hold']).cumprod(), label='EW Mag 7 buy & hold', color='black', ls='--', lw=1)
    ax.plot(port.index, (1 + port['sp500']).cumprod(), label='S&P 500', color='grey', lw=1)
    if 'sentiment_era' in EVAL_WINDOWS:          # only when it differs from the first OOS date
        ax.axvline(pd.Timestamp(SENTIMENT_EVAL_START), color='grey', ls=':', lw=1)
    ax.set_yscale('log'); ax.set_title(f'{arch}: OOS equity curve'); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig('equity_curves_oos.png', dpi=120, bbox_inches='tight'); plt.show()

_windows_with_data = [w for w in EVAL_WINDOWS
                      if not all_bt[(all_bt.ticker == 'POOLED') & (all_bt.series == 'strategy') & (all_bt.window == w)].empty]
fig, axes = plt.subplots(1, len(_windows_with_data), figsize=(7 * len(_windows_with_data), 4.5), squeeze=False)
for ax, w in zip(axes[0], _windows_with_data):
    piv = all_bt[(all_bt.ticker == 'POOLED') & (all_bt.series == 'strategy') & (all_bt.window == w)].pivot_table(
        index='architecture', columns='feature_set', values='sharpe')[FS_ORDER]
    piv = piv.loc[[a for a in ALL_RESULTS if a in piv.index]]        # Ridge LogReg, XGBoost, LSTM (pivot sorts alphabetically)
    piv.plot(kind='bar', ax=ax)                                      # default colours: same blue/orange/green as the curves
    ax.axhline(0, color='grey', lw=1); ax.set_ylabel('Sharpe ratio')
    ax.set_title(f'Sharpe by architecture x feature set ({w})' + (' [QUICK_VERIFY]' if QUICK_VERIFY else ''))
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.savefig('sharpe_by_architecture.png', dpi=120, bbox_inches='tight'); plt.show()

In [ ]:
# RETURN GRAPH -- cumulative return by architecture x feature set, read from the backtest results (nothing typed in by hand)
_p = all_bt[all_bt.ticker == 'POOLED']
_strat = _p[_p.series == 'strategy']
_colors = {'baseline': '#4472C4', 'finbert': '#ED7D31', 'qwen': '#70AD47'}
_wins = list(EVAL_WINDOWS)
fig, axes = plt.subplots(1, len(_wins), figsize=(8 * len(_wins), 6), squeeze=False)
for ax, w in zip(axes[0], _wins):
    piv = _strat[_strat.window == w].pivot_table(index='architecture', columns='feature_set', values='cumulative_return')
    piv = piv.reindex(index=list(ALL_RESULTS), columns=FS_ORDER) * 100
    x, width = np.arange(len(piv)), 0.26
    for i, fs in enumerate(FS_ORDER):
        bars = ax.bar(x + i * width, piv[fs].values, width, label=fs.capitalize(), color=_colors[fs], alpha=0.85, edgecolor='white')
        for bar, val in zip(bars, piv[fs].values):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{val:,.0f}%', ha='center', va='bottom', fontsize=8)
    top = float(np.nanmax(piv.values)) * 1.25
    ax.set_ylim(0, top)
    for s_, ls, lab in [('ew_buy_hold', '--', 'EW Mag 7 buy & hold'), ('sp500', ':', 'S&P 500')]:
        val = float(_p[(_p.series == s_) & (_p.window == w)]['cumulative_return'].iloc[0]) * 100
        if val <= top:
            ax.axhline(val, color='black' if s_ == 'ew_buy_hold' else 'grey', ls=ls, lw=1, label=f'{lab} ({val:,.0f}%)')
        else:
            ax.plot([], [], ' ', label=f'{lab}: {val:,.0f}% (off scale)')
    start = pd.Timestamp(FIRST_OOS_DATE if EVAL_WINDOWS[w] is None else EVAL_WINDOWS[w]).year
    ax.set_xticks(x + width); ax.set_xticklabels(piv.index); ax.set_ylabel('Cumulative return (%)')
    ax.set_title(f"{'Full out-of-sample' if w == 'full_oos' else 'Sentiment era'} ({start}-{pd.Timestamp(DATA_END).year})\nCumulative return of the top-{PORTFOLIO_TOP_K} strategy (%)", fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
fig.suptitle('Can sentiment improve returns? Baseline vs FinBERT vs Qwen' + (' [QUICK_VERIFY]' if QUICK_VERIFY else ''), fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig('return_comparison.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# =====================================================================
# Significance test on custom sub-periods (reuses the existing walk-forward predictions, no retraining)
# Run after results_logreg / results_xgb / results_lstm and the significance functions are defined.
# =====================================================================
from statsmodels.stats.multitest import multipletests

# ---- Settings ------------------------------------------------------
PERIODS = [   # (label, start, end) -- inclusive dates on the rebalance calendar
    ('Full OOS (reproduces earlier table)', SENTIMENT_EVAL_START, '2025-12-31'),
    ('2016-2020', '2016-01-04', '2020-12-31'),
    ('2021-2025', '2021-01-04', '2025-12-31'),
]
BLOCK_SIZES = [BLOCK_SIZE]        # e.g. [1, 2, 4, 10, 13] for block-length sensitivity
BREAKERS    = [True]              # [True, False] to also test without the drawdown breaker
N_PERM_USE  = 2000
# --------------------------------------------------------------------

ARCHS = {'Ridge LogReg': results_logreg, 'XGBoost': results_xgb, 'LSTM': results_lstm}
COMPARISONS = [('baseline', 'finbert'), ('baseline', 'qwen'), ('finbert', 'qwen')]

def period_returns(res, start, end, apply_breaker=True):
    """Pooled weekly strategy returns for each feature set, restricted to [start, end]."""
    out = {}
    for nm in res['proba_by_model']:
        meta, proba = res['meta_by_model'][nm], res['proba_by_model'][nm]
        mask = ((meta['date'] >= pd.Timestamp(start)) & (meta['date'] <= pd.Timestamp(end))).values
        out[nm] = compute_pooled_return_series(proba[mask], meta[mask].reset_index(drop=True),
                                               HORIZON, apply_breaker=apply_breaker)
    return out

rows = []
for label, start, end in PERIODS:
    for breaker in BREAKERS:
        for arch, res in ARCHS.items():
            ret = period_returns(res, start, end, apply_breaker=breaker)
            for a, b in COMPARISONS:
                for bs in BLOCK_SIZES:
                    ci   = bootstrap_paired_return_difference(ret[a], ret[b], block_size=bs)
                    perm = permutation_test_return_difference(ret[a], ret[b], block_size=bs, n_perm=N_PERM_USE)
                    rows.append(dict(period=label, start=start, end=end, breaker=breaker, block=bs,
                                     architecture=arch, comparison=f'{a} vs {b}',
                                     n_periods=ci['n_periods'],
                                     active_a=int((ret[a] != 0).sum()), active_b=int((ret[b] != 0).sum()),
                                     return_a=ci['cum_return_1'] * 100, return_b=ci['cum_return_2'] * 100,
                                     diff_pp=ci['point'] * 100, ci_lo=ci['ci_lower'] * 100, ci_hi=ci['ci_upper'] * 100,
                                     p_value=perm['p_value']))

sig_period = pd.DataFrame(rows)

# Holm correction within each family of 9 tests (same period / breaker / block length)
sig_period['p_holm'] = np.nan
for _, g in sig_period.groupby(['period', 'breaker', 'block']):
    sig_period.loc[g.index, 'p_holm'] = multipletests(g['p_value'].fillna(1.0), method='holm')[1]
sig_period['significant_5pct'] = sig_period['p_value'] < 0.05

print(f"Block size(s): {BLOCK_SIZES} | permutations: {N_PERM_USE} | breaker: {BREAKERS}")
print(sig_period.round(3).to_string(index=False))
print("\nSignificant at 5% (unadjusted):", int(sig_period['significant_5pct'].sum()), "of", len(sig_period))
print("Significant after Holm (5%):   ", int((sig_period['p_holm'] < 0.05).sum()), "of", len(sig_period))
low = sig_period[sig_period['n_periods'] < 60]
if len(low):
    print(f"\nWARNING: {len(low)} row(s) have fewer than 60 weekly periods (very low power / few blocks).")
sig_period.to_csv('three_family_significance_by_period.csv', index=False)

In [ ]:
# Equity curves without the equal-weight line (for the main report)
fig, axes = plt.subplots(1, len(ALL_RESULTS), figsize=(6 * len(ALL_RESULTS), 4.5), sharey=False, squeeze=False)
for ax, (arch, res) in zip(axes[0], ALL_RESULTS.items()):
    for fs in FS_ORDER:
        port = pooled_series(build_book(res['proba_by_model'][fs], res['meta_by_model'][fs], HORIZON))
        ax.plot(port.index, (1 + port['strategy_return']).cumprod(), label=fs, lw=1.2)
    ax.plot(port.index, (1 + port['sp500']).cumprod(), label='S&P 500', color='grey', lw=1)
    ax.set_yscale('log'); ax.set_title(f'{arch}: OOS equity curve'); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig('equity_curves_oos_no_ew.png', dpi=120, bbox_inches='tight'); plt.show()